# 16: Paper results: gates, final tables, figures and claims check

Version 2 (manuscript stage) adds: the survey-weighted sensitivity analysis of plan Section 6.2 for BRFSS Heart disease and CDC Diabetes (survey weights of the rebuilt 2015 table, which equals the public table row by row), the explanation cards of the manuscript introduction (the box released to the most explained test patients for each task and claim kind, from the saved releases), manuscript Figures 1 and 2 (schematic), the manuscript figure numbering in results/16_figures.csv, and corrected wording of claim V4 now that no configuration is inconclusive.

Design decisions
- Reads only committed results files, the frozen configurations, the public tables (recorded outcomes of the explained test patients, for the bootstrap), the candidate caches of notebooks 08 and 09d on Drive (candidate statistics of the frozen models) and the NHANES table of notebook 03 (one descriptive figure). No model is fitted and no certificate is recomputed; every data file is checked against the hash recorded when its results were made.
- Gates G0 to G5 from the results files (G4 also uses results/10b_validity_more_repetitions.csv when present).
- Main table (E1): BRFSS Heart disease, CDC Diabetes, Cardio-70k and UCI Heart with XGBoost, LightGBM and random forest at the frozen targets, negative-outcome and enrichment claims in separate rows. UCI Heart pools the four held-out sites (frozen heart targets, no policy test); per-site rows are in a second table.
- Intervals: 95% percentile bootstrap over the explained test patients, 10,000 resamples per model. Each statistic depends on a patient only through five categories (no release; negative-outcome release correct or wrong; enrichment release correct or wrong), so resampling patients equals drawing the category counts from a multinomial. XGBoost uses the saved per-patient releases with outcomes from the public tables (they must reproduce the saved counts); LightGBM and random forest, whose per-patient releases were not saved, use the saved counts; the notebook checks on the XGBoost rows that both give the same distribution. UCI resamples within each site. The one-sided 95% Clopper-Pearson bounds of the earlier notebooks are kept beside the intervals.
- Every figure is regenerated from the results CSV files at 600 dpi under its existing name, plus two new paper figures (plan Figure 4, validity against delta_1; plan Figure 5, certification-rate curves for three tasks and three families). Fixes: the subgroup legend lists the policy-target line only when it is drawn; the within-box shift figure has whole-number percentage ticks. Plan Figures 1 and 2 are drawn for the manuscript.
- The claims register (results/16_claims_check.md) states every numerical claim planned for the paper with the results file it comes from and a check computed from that file.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver', 'statsmodels'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, math, pickle, platform
from importlib import metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import (FixedFormatter, FixedLocator, FuncFormatter, LogLocator, MaxNLocator, NullFormatter,
                               NullLocator, PercentFormatter)
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.frozen import load_frozen
from scax.data import public, uci
from scax.experiments import paper
from scax.experiments.paper import md_table, pct, pct_ci

SEED, B = 42, 10000            # bootstrap resamples per model
STRICT = True                  # data files must match the hashes recorded when the results were made
seed_everything(SEED)
RES, FIG = paths.RESULTS, paths.FIGURES
FIG.mkdir(parents=True, exist_ok=True)


def rd(name):
    return pd.read_csv(RES / name)


def rj(name):
    return json.load(open(RES / name))


def check_data(path, expected, what):
    h = sha256_file(path)
    if h != expected:
        msg = f'{what}: {path.name} differs from the file used for the results ({h[:12]} against {expected[:12]})'
        if STRICT:
            raise ValueError(msg)
        print('WARNING (test run):', msg)
    return h == expected


TASKS = ['brfss2015_heart', 'cdc_diabetes', 'cardio70k', 'uci_heart']
TASK_NAMES = {'brfss2015_heart': 'BRFSS Heart disease', 'cdc_diabetes': 'CDC Diabetes', 'cardio70k': 'Cardio-70k',
              'uci_heart': 'UCI Heart'}
FAMILIES = ['xgboost', 'lightgbm', 'sklearn_rf']
FAMILY_NAMES = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM', 'sklearn_rf': 'Random forest'}
KIND = {0: 'negative outcome', 1: 'enrichment'}
SHA = {**rj('05_frozen.json')['frozen_config_sha256'], **rj('09c_frozen.json')['frozen_config_sha256']}
FROZEN = {t: load_frozen(paths.REPO_DIR / 'configs' / 'frozen' / f'{t}.json', SHA[t])[0]
          for t in ('cdc_diabetes', 'brfss2015_heart', 'cardio70k')}
TARGETS = {t: c['targets'] for t, c in FROZEN.items()}
for t, T in TARGETS.items():
    print(f"{TASK_NAMES[t]}: alpha_0 {T['alpha_0']}, beta {T['beta']} (alpha_1 {T['alpha_1']}), alpha_sel {T['alpha_sel']}")

## Inputs: results manifest, recorded versions, pinned requirements

In [ ]:
INPUTS = sorted(p for p in RES.iterdir() if p.is_file() and p.suffix in ('.csv', '.json') and not p.name.startswith('16_'))
MANIFEST = pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in INPUTS])
VERSIONS = []
for p in INPUTS:
    if p.suffix == '.json':
        d = json.load(open(p))
        if d.get('_versions'):
            VERSIONS.append({'file': p.name, 'written_utc': d.get('_written_utc'), **d['_versions']})
VERSIONS = pd.DataFrame(VERSIONS)
names = [l.split('==')[0].strip() for l in open(paths.REPO_DIR / 'requirements.txt') if l.strip() and not l.startswith('#')]
lock = []
for name in names:
    try:
        lock.append(f'{name}=={metadata.version(name)}')
    except metadata.PackageNotFoundError:
        lock.append(f'# {name}: not installed in this session')
LOCK_TEXT = (f'# Package versions in the Colab session that ran notebook 16 (Python {platform.python_version()}).\n'
             '# The versions behind each result are recorded in its results file (_versions); see results/16_versions.csv.\n'
             + '\n'.join(lock) + '\n')
print(f'{len(INPUTS)} results files; versions recorded in {len(VERSIONS)} JSON files')
print(VERSIONS.drop(columns=['file', 'written_utc']).drop_duplicates().to_string(index=False))

## Decision gates G0 to G5

In [ ]:
GATES = []


def gate(g, rule, evidence, status, sources):
    GATES.append({'gate': g, 'rule': rule, 'evidence': evidence, 'status': status, 'sources': sources})


g0 = rj('02_brfss_rebuild.json')['gate_G0']
gate('G0 data', 'The rebuilt BRFSS 2015 table equals the public table',
     f"{g0['rows_rebuilt']:,} rebuilt rows against {g0['rows_public']:,}; equal row by row: {g0['row_by_row_equal']}; "
     f"marginal mismatches: {len(g0['marginal_mismatches'])}; diabetes outcome equal: {g0['diabetes_task_outcome_equal']}",
     'passed' if g0['passed'] else 'failed', '02_brfss_rebuild.json')

dev = pd.concat([rd('05_dev_configs.csv'), rd('09c_dev_configs.csv')], ignore_index=True)
G1 = []
for t, c in FROZEN.items():
    r = dev[(dev.task == t) & (dev.config == c['model']['config'])].iloc[0]
    G1.append({'task': t, 'auroc': float(r.auroc_dev_cal), 'reference': float(r.g1_reference), 'passed': bool(r.g1_pass)})
gate('G1 model', 'The frozen model loses at most 0.01 AUROC against a tuned unconstrained model (development)',
     '; '.join(f"{TASK_NAMES[g['task']]}: {g['auroc']:.4f} against {g['reference']:.4f}" for g in G1),
     'passed' if all(g['passed'] for g in G1) else 'failed', '05_dev_configs.csv, 09c_dev_configs.csv')

# G2: every disagreement counter recorded anywhere; budget overruns (unknown) remove boxes and are reported
g2 = rj('06_encoding_tests.json')['gate_G2']
DIS = {'notebook 06, development models': g2['exported_vs_native_disagreements_beyond_1e-5_of_tau'] + g2['primary_vs_second_disagreements']
       + g2['precheck_wrong'] + g2['z3_disagreements'] + g2['brute_force_errors']}
EV = [f"notebook 06: exported against native {g2['exported_vs_native_disagreements']}, primary against second check "
      f"{g2['primary_vs_second_disagreements']}, pre-check wrong {g2['precheck_wrong']}, Z3 disagreements {g2['z3_disagreements']} "
      f"of {g2['z3_checked']} ({g2['z3_unknown']} unknown), brute-force errors {g2['brute_force_errors']} of {g2['brute_force_checks']:,}"]
for t, c in FROZEN.items():
    fm = c['frozen_model_checks']; sc = fm['soundness_checks']
    env = fm['exported_vs_native']['beyond_rounding'] if 'exported_vs_native' in fm else fm['exported_vs_native_disagreements']
    DIS[f'{t} frozen model'] = env + sc['primary_vs_second_disagree'] + sc['z3_disagree']
    EV.append(f"{TASK_NAMES[t]} frozen model: exported against native {env}; {sc['boxes']} boxes, primary against second check "
              f"{sc['primary_vs_second_disagree']}; Z3 {sc['z3_disagree']} disagreements of {sc['z3_checked']}")
VER = {**rj('08_certificates.json')['verification'], 'cardio70k': rj('09d_results.json')['verification']}
for t, v in VER.items():
    EV.append(f"{TASK_NAMES[t]}: {v['sound']:,} of {v['eligible_boxes']:,} releasable boxes sound by the second exact check "
              f"({v['unknown']} unknown); Z3 sample {v['z3_sample']['sound']} sound, {v['z3_sample']['unknown']} unknown")
UNITS = (list(rj('09b_families.json')['units'].values()) + list(rj('09e_families.json')['cardio'].values())
         + list(rj('09e_families.json')['uci'].values()))
DIS['LightGBM and random forest'] = sum(u.get('exported_vs_native_training', {}).get('beyond_rounding', 0) for u in UNITS)
EV.append(f"LightGBM and random forest ({len(UNITS)} models): exported against native beyond rounding {DIS['LightGBM and random forest']}; "
          f"releasable boxes left unverified {sum(u.get('unverified_boxes', 0) for u in UNITS)}")
SM = list(rj('13_shift.json')['models'].values()) + list(rj('13b_shift.json')['models'].values())
beyond = lambda m: m['exported_vs_native']['beyond_rounding'] if isinstance(m.get('exported_vs_native'), dict) else m.get('exported_vs_native_disagreements', 0)
within = sum(m['exported_vs_native'].get('within_rounding', 0) for m in SM if isinstance(m.get('exported_vs_native'), dict))
DIS['shift models'] = sum(beyond(m) for m in SM)
EV.append(f"shift models ({len(SM)}): exported against native beyond rounding {DIS['shift models']}; rows within float32 rounding of "
          f"the threshold, decided by the exact sum: {within}")
V12 = rd('12b_model_variants.csv')
DIS['ablation variants'] = int(V12.exported_vs_native_disagreements.sum() + V12.second_check_failures.sum())
EV.append(f"ablation variants ({len(V12)}): exported against native {int(V12.exported_vs_native_disagreements.sum())}, "
          f"second-check failures {int(V12.second_check_failures.sum())}")
S10 = rd('10_settings.csv')
EV.append(f"synthetic population models ({len(S10)}): unverified boxes removed {int(S10.unverified_removed.sum())}, searches "
          f"stopped by the budget {int(S10.search_unknown.sum())}")
gate('G2 soundness', 'No disagreement between encodings, brute force, the two exact checks, or exported and native predictions',
     ' | '.join(EV), 'passed' if sum(DIS.values()) == 0 else 'failed',
     '06_encoding_tests.json, configs/frozen/*.json, 08_certificates.json, 09d_results.json, 09b_families.json, 09e_families.json, '
     '13_shift.json, 13b_shift.json, 12b_model_variants.csv, 10_settings.csv')

g3 = {t: c['development'] for t, c in FROZEN.items()}
short = [TASK_NAMES[t] for t, d in g3.items() if not d['target_met']]
gate('G3 utility (development only)', 'At least half of the development patients get a negative-outcome explanation at the main '
     'target; otherwise the development cascade, then freezing with the shortfall recorded',
     '; '.join(f"{TASK_NAMES[t]}: {d['release_rate_negative']:.1%} (target met: {d['target_met']})" for t, d in g3.items()),
     'passed' if not short else f"passed for the other tasks; {', '.join(short)}: cascade followed, shortfall recorded before "
     'calibration split 1 was opened (docs/PLAN.md)', 'configs/frozen/*.json')

V = rd('10_validity.csv')
KEYS = ['rule', 'eps', 'n1', 'delta1', 'M', 'prior']
MORE = RES / '10b_validity_more_repetitions.csv'
V['V1_reps'] = V.reps                          # box-certificate repetitions (more where notebook 10b added them)
if MORE.exists():
    X = pd.read_csv(MORE)
    V = V.merge(X[KEYS + ['reps_total', 'V1_total', 'V1_rate_total', 'V1_upper_total', 'V1_verdict_total']], on=KEYS, how='left')
    upd = V.reps_total.notna()
    for a, b in (('V1_reps', 'reps_total'), ('V1', 'V1_total'), ('V1_rate', 'V1_rate_total'), ('V1_upper', 'V1_upper_total'),
                 ('V1_verdict', 'V1_verdict_total')):
        V.loc[upd, a] = V.loc[upd, b]
    V = V.drop(columns=[c for c in V.columns if c.endswith('_total')])
    print(f'{int(upd.sum())} configurations updated with the repetitions of notebook 10b')
VS = V[V.prior != 'naive']
c1, c2 = VS.V1_verdict.value_counts().to_dict(), VS.V2_verdict.value_counts().to_dict()
excess, inconc = c1.get('excess', 0) + c2.get('excess', 0), VS[(VS.V1_verdict == 'inconclusive') | (VS.V2_verdict == 'inconclusive')]
near = bool((inconc.V1_rate <= inconc.delta1 + 1e-12).all()) if len(inconc) else True
g4_status = ('failed: excess violations' if excess else 'passed' if not len(inconc) else
             f'no excess; {len(inconc)} inconclusive, every point estimate at or below delta_1' + (
                 ' (after the added repetitions of notebook 10b)' if MORE.exists() else ' (plan: add repetitions, notebook 10b)'))
gate('G4 validity', 'Three-way rule with multiplicity correction: no configuration with excess violations; inconclusive ones get more repetitions',
     f"{len(VS)} SCAX configurations (two priors): box certificates {c1}; policy {c2}; largest violation frequency "
     f"{VS.V1_rate.max():.3f} (box) and {VS.V2_rate.max():.3f} (policy); box-certificate repetitions per configuration {int(VS.V1_reps.min())} to {int(VS.V1_reps.max())}",
     g4_status, '10_validity.csv' + (', 10b_validity_more_repetitions.csv' if MORE.exists() else ''))

g5 = rj('15_runtime.json')['G5']
med = g5['production_median_seconds']
gate('G5 runtime', 'Median above 30 seconds per distinct cell for a frozen model triggers changes',
     f"production medians {min(med.values()):.2f} to {max(med.values()):.2f} seconds per cell ({len(med)} models)",
     'passed' if g5['passed'] else 'failed', '15_runtime.json')
GATES = pd.DataFrame(GATES)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 400)
for r in GATES.itertuples():
    print(f'{r.gate}: {r.status}\n    {r.evidence}\n')

## Main table (E1) with 95% bootstrap intervals over test patients

In [ ]:
Y, DATA_OK = {}, {}
for t in ('cdc_diabetes', 'brfss2015_heart', 'cardio70k'):
    DATA_OK[t] = check_data(public.raw_dir(paths.RAW, t) / public.SPECS[t].file, FROZEN[t]['data_sha256'], TASK_NAMES[t])
    Y[t] = np.asarray(public.LOADERS[t](paths.RAW)[1], dtype=int)
REL = {t: d[d.task == t] for d in (rd('09_test_releases.csv'), rd('09d_test_releases.csv')) for t in d.task.unique()}
SOURCES = {'cdc_diabetes': ('09b_main.csv', '09b_model_metrics.csv'), 'brfss2015_heart': ('09b_main.csv', '09b_model_metrics.csv'),
           'cardio70k': ('09e_cardio_main.csv', '09e_cardio_model_metrics.csv')}
MAIN, CHECK = [], []


def add_rows(task, fmy, rows, s, method, metrics, source, note=''):
    for k in (0, 1):
        r = rows[k]
        MAIN.append({'task': task, 'task_name': TASK_NAMES[task], 'family': fmy, 'family_name': FAMILY_NAMES[fmy], 'kind': KIND[k],
                     'claim_target': r['target'], 'alpha_sel': r.get('alpha_sel'), 'cal2_accepted': r.get('cal2_accepted'),
                     'cal2_errors': r.get('cal2_errors'), 'U_sel': r.get('U_sel'), 'policy_certified': r.get('policy_certified'),
                     'test_patients': r['patients'], 'test_accepted': r['accepted'], 'certified_share': s[f'share_{k}'],
                     'certified_share_lo': s[f'share_{k}_lo'], 'certified_share_hi': s[f'share_{k}_hi'], 'test_errors': r['errors'],
                     'error_rate': s[f'error_{k}'], 'error_rate_lo': s[f'error_{k}_lo'], 'error_rate_hi': s[f'error_{k}_hi'],
                     'error_upper_95_cp': r.get('error_upper_95'), 'mean_size': r.get('mean_size'),
                     'median_support': r.get('median_support'), 'auroc': metrics.get('auroc'), 'auprc': metrics.get('auprc'),
                     'brier': metrics.get('brier'), 'ece': metrics.get('ece_15_equal_mass'), 'interval': method,
                     'note': note, 'source': source})


for task in ('brfss2015_heart', 'cdc_diabetes', 'cardio70k'):
    mf, mm = SOURCES[task]
    M = rd(mf)
    M = M[(M.task == task) & (M.frozen_targets == True)]
    if 'grid' in M:
        M = M[M.grid == 'scaled']
    MM = rd(mm); MM = MM[MM.task == task]
    for fmy in FAMILIES:
        d = M[M.family == fmy].set_index('kind')
        rows = {}
        for k in (0, 1):
            r = d.loc[KIND[k]]
            rows[k] = {'target': float(r.target), 'alpha_sel': float(r.alpha_sel), 'cal2_accepted': int(r.cal2_accepted),
                       'cal2_errors': int(r.cal2_errors), 'U_sel': float(r.U_sel), 'policy_certified': bool(r.policy_certified),
                       'patients': int(r.test_patients), 'accepted': int(r.test_accepted), 'errors': int(r.test_errors),
                       'error_upper_95': float(r.test_error_upper_95), 'mean_size': float(r.mean_size),
                       'median_support': float(r.median_calibration_support)}
        obs = paper.counts_from(rows[0]['patients'], rows[0]['accepted'], rows[0]['errors'], rows[1]['accepted'], rows[1]['errors'])
        seed = paper.label_seed(f'{task}/{fmy}')
        s_counts = paper.summarise(paper.bootstrap_counts(obs, B, seed), obs)
        s, method = s_counts, 'bootstrap over patients, from the saved counts'
        if fmy == 'xgboost':
            rel = REL[task]
            cat = paper.categories(rel.predicted, rel.released, Y[task][rel.row.to_numpy()])
            assert (np.bincount(cat, minlength=5) == obs).all(), f'{task}: saved releases do not reproduce the saved counts'
            s = paper.summarise(paper.bootstrap_patients(cat, B, seed), obs)
            method = 'bootstrap over patients, from the saved releases'
            CHECK.append({'task': task, 'family': fmy, 'patients': len(rel), 'same_distribution': paper.same_distribution(s, s_counts, B),
                          **{f'{k}_{w}_{src}': ss[f'{k}_{w}'] for src, ss in (('releases', s), ('counts', s_counts))
                             for k in ('share_0', 'share_1', 'error_0', 'error_1') for w in ('lo', 'hi')}})
        met = MM[MM.family == fmy].iloc[0].to_dict()
        add_rows(task, fmy, rows, s, method, met, f'{mf}, {mm}' + (', 09_test_releases.csv' if task != 'cardio70k' and fmy == 'xgboost'
                                                              else ', 09d_test_releases.csv' if fmy == 'xgboost' else ''))

# UCI Heart: the four held-out sites pooled (frozen heart targets), resampled within each site
U = rd('09e_uci_main.csv')
UT = U[(U.role == 'target') & (U.targets_set == 'frozen')]
for fmy in FAMILIES:
    d = UT[UT.family == fmy]
    strata, acc, err, size_sum, tg = [], {0: 0, 1: 0}, {0: 0, 1: 0}, {0: 0.0, 1: 0.0}, {}
    for model, g in d.groupby('model'):
        g = g.set_index('kind')
        n = int(g.loc[0, 'explained'])
        a = {k: int(g.loc[k, 'accepted']) for k in (0, 1)}
        e = {k: int(np.nan_to_num(g.loc[k, 'errors'])) for k in (0, 1)}
        strata.append(paper.counts_from(n, a[0], e[0], a[1], e[1]))
        for k in (0, 1):
            acc[k] += a[k]; err[k] += e[k]; tg[k] = float(g.loc[k, 'alpha_claim'])
            size_sum[k] += a[k] * float(np.nan_to_num(g.loc[k, 'mean_size']))
    obs = np.sum(strata, axis=0)
    s = paper.summarise(paper.bootstrap_counts(strata, B, paper.label_seed(f'uci_heart/{fmy}')), obs)
    rows = {k: {'target': tg[k], 'patients': int(obs.sum()), 'accepted': acc[k], 'errors': err[k],
                'mean_size': size_sum[k] / acc[k] if acc[k] else None} for k in (0, 1)}
    add_rows('uci_heart', fmy, rows, s, 'bootstrap over patients, from the saved counts, within each site', {}, '09e_uci_main.csv',
             note=f'{len(strata)} held-out sites pooled; no policy test; per-site model metrics in 16_table_uci_sites.csv')
MAIN = pd.DataFrame(MAIN)
# the share with the outcome among released patients: the error for negative-outcome claims, one minus it for enrichment
neg = MAIN.kind == 'negative outcome'
MAIN['outcome_share'] = np.where(neg, MAIN.error_rate, 1 - MAIN.error_rate)
MAIN['outcome_share_lo'] = np.where(neg, MAIN.error_rate_lo, 1 - MAIN.error_rate_hi)
MAIN['outcome_share_hi'] = np.where(neg, MAIN.error_rate_hi, 1 - MAIN.error_rate_lo)
MAIN['outcome_bound_95_cp'] = np.where(neg, MAIN.error_upper_95_cp, 1 - MAIN.error_upper_95_cp.astype(float))
CHECK = pd.DataFrame(CHECK)
assert CHECK.same_distribution.all(), 'bootstrap from releases and from counts disagree'
print(f'bootstrap from releases and from counts agree for {len(CHECK)} models')

UCI_SITES = U[U.role == 'target'].assign(site=lambda d: d.target.str.replace(' held out', '', regex=False))[
    ['site', 'family_name', 'targets_set', 'kind_name', 'alpha_claim', 'explained', 'accepted', 'coverage', 'coverage_lo',
     'coverage_hi', 'errors', 'error_rate', 'error_upper_95', 'refuted_share', 'auroc', 'prevalence']]
show = MAIN.assign(target=MAIN.claim_target.map(pct), share=[pct_ci(*v) for v in MAIN[['certified_share', 'certified_share_lo', 'certified_share_hi']].values],
                   with_outcome=[pct_ci(*v) for v in MAIN[['outcome_share', 'outcome_share_lo', 'outcome_share_hi']].values])
print(show[['task_name', 'family_name', 'kind', 'target', 'policy_certified', 'share', 'with_outcome']].to_string(index=False))

## Survey-weighted sensitivity analysis (plan Section 6.2) and explanation cards

In [ ]:
# BRFSS Heart disease and CDC Diabetes share the 253,680 respondents of the 2015 table; the rebuilt 2015 table (notebook 02)
# holds the survey weight and equals the public table row by row (gate G0), checked again here
WEIGHTED, BR = [], paths.DERIVED / 'brfss' / 'brfss_2015.parquet'
if BR.exists():
    check_data(BR, rj('02_brfss_rebuild.json')['parquet_sha256']['2015'], 'rebuilt BRFSS 2015 table')
    T15 = pd.read_parquet(BR)
    Xh, yh = public.LOADERS['brfss2015_heart'](paths.RAW)
    cols = list(Xh.columns)
    assert len(T15) == len(Xh) and np.array_equal(T15[cols].to_numpy(float), Xh[cols].to_numpy(float)) and \
        np.array_equal(T15['HeartDiseaseorAttack'].to_numpy(float), np.asarray(yh, float)), 'rebuilt table differs from the public table'
    W = T15['_LLCPWT'].to_numpy(float)
    assert np.isfinite(W).all() and (W > 0).all()
    for task in ('brfss2015_heart', 'cdc_diabetes'):
        rel = REL[task]
        rows = rel.row.to_numpy()
        cat = paper.categories(rel.predicted, rel.released, Y[task][rows])
        w = W[rows]
        sw = paper.summarise(paper.bootstrap_patients(cat, B, paper.label_seed(f'{task}/weighted'), weights=w), paper.weighted_counts(cat, w))
        for k in (0, 1):
            m = MAIN[(MAIN.task == task) & (MAIN.family == 'xgboost') & (MAIN.kind == KIND[k])].iloc[0]
            e, elo, ehi = sw[f'error_{k}'], sw[f'error_{k}_lo'], sw[f'error_{k}_hi']
            WEIGHTED.append({'task': task, 'task_name': TASK_NAMES[task], 'kind': KIND[k], 'claim_target': m.claim_target,
                             'alpha_sel': m.alpha_sel, 'patients': len(rel),
                             'certified_share': m.certified_share, 'certified_share_weighted': sw[f'share_{k}'],
                             'certified_share_weighted_lo': sw[f'share_{k}_lo'], 'certified_share_weighted_hi': sw[f'share_{k}_hi'],
                             'outcome_share': m.outcome_share,
                             'outcome_share_weighted': e if k == 0 else 1 - e,
                             'outcome_share_weighted_lo': elo if k == 0 else 1 - ehi,
                             'outcome_share_weighted_hi': ehi if k == 0 else 1 - elo,
                             'prevalence_table': float(np.mean(Y[task])), 'prevalence_table_weighted': float(np.average(Y[task], weights=W)),
                             'prevalence_explained_test': float(Y[task][rows].mean()),
                             'prevalence_explained_test_weighted': float(np.average(Y[task][rows], weights=w))})
    WEIGHTED = pd.DataFrame(WEIGHTED)
    print(WEIGHTED.round(4).T.to_string())
else:
    WEIGHTED = pd.DataFrame()
    print('rebuilt BRFSS 2015 table not on Drive; weighted sensitivity analysis skipped')

# explanation cards: for each task and claim kind, the box released to the most explained test patients (saved releases)
CARDS = []
for task in ('brfss2015_heart', 'cdc_diabetes', 'cardio70k'):
    rel = REL[task]
    for k in (0, 1):
        g = rel[rel.released & (rel.predicted == k)]
        if not len(g):
            continue
        top = g.groupby(['explanation', 'size', 'n_B', 'U']).size().sort_values(ascending=False, kind='stable')
        (expl, size, nB, Ub), cnt = top.index[0], int(top.iloc[0])
        tgt = TARGETS[task]['alpha_0'] if k == 0 else TARGETS[task]['alpha_1']
        inside = g.explanation == expl
        yy = Y[task][g.row.to_numpy()[inside.to_numpy()]]
        CARDS.append({'task': task, 'task_name': TASK_NAMES[task], 'kind': KIND[k], 'explanation': expl, 'features': int(size),
                      'calibration_records': int(nB), 'bound': float(Ub), 'claim_target': tgt,
                      'claim': (f'at most {pct(Ub)} have the outcome (target at most {pct(tgt)})' if k == 0
                                else f'at least {pct(1 - Ub)} have the outcome (target at least {pct(1 - tgt)})'),
                      'explained_test_patients_given_it': cnt, 'outcome_share_among_them': float(np.mean(yy))})
CARDS = pd.DataFrame(CARDS)
print(CARDS.drop(columns=['task']).to_string(index=False))

## Candidate statistics of the frozen models (candidate caches of notebooks 08 and 09d)

In [ ]:
CAND, CERT = [], paths.DERIVED / 'certificates'
BOX_SHA = {**rj('08_certificates.json')['box_table_sha256'], 'cardio70k': rj('09d_results.json')['box_table_sha256']}
for t in ('cdc_diabetes', 'brfss2015_heart', 'cardio70k'):
    f, fb = CERT / f'{t}__candidates.pkl', CERT / f'{t}__boxes.parquet'
    if not (f.exists() and fb.exists()):
        print(f'{TASK_NAMES[t]}: candidate cache not on Drive, skipped'); continue
    check_data(fb, BOX_SHA[t], f'{TASK_NAMES[t]} box table')
    with open(f, 'rb') as fh:
        d = pickle.load(fh)
    bx = pd.read_parquet(fb)
    T = TARGETS[t]; alpha = {0: T['alpha_0'], 1: T['alpha_1']}; Mx = FROZEN[t]['candidates']['M']
    idx = {(k, int(c)): i for i, (k, c) in enumerate(zip(bx.key, bx.box_class))}
    U_, nB, sound = bx.U.to_numpy(), bx.n_B.to_numpy(), bx.verified.to_numpy() == 'sound'
    per, missing = [], 0
    for key, cands in d['cache'].items():
        cs = [c for c in cands if c.order < Mx]
        ii = [idx.get((json.dumps(c.key), int(c.y))) for c in cs]
        if any(i is None for i in ii):
            missing += 1; continue
        if not cs:
            per.append({'candidates': 0, 'releasable': False, 'seconds': d['seconds'].get(key)}); continue
        ii = np.array(ii)
        per.append({'candidates': len(cs), 'axps': sum(c.kind == 'axp' for c in cs), 'smallest_size': min(c.size for c in cs),
                    'largest_size': max(c.size for c in cs), 'largest_support': int(nB[ii].max()), 'smallest_bound': float(U_[ii].min()),
                    'releasable': bool(((U_[ii] <= alpha[cs[0].y]) & sound[ii]).any()), 'seconds': d['seconds'].get(key)})
    P = pd.DataFrame(per)
    CAND.append({'task': t, 'task_name': TASK_NAMES[t], 'distinct_cells': len(P), 'cells_without_box_rows': missing,
                 'candidates_median': P.candidates.median(), 'candidates_q25': P.candidates.quantile(0.25),
                 'candidates_q75': P.candidates.quantile(0.75), 'axp_share_of_candidates': P.axps.sum() / P.candidates.sum(),
                 'smallest_size_median': P.smallest_size.median(), 'largest_size_median': P.largest_size.median(),
                 'largest_support_median': P.largest_support.median(), 'cells_with_releasable_box': P.releasable.mean(),
                 'distinct_boxes': len(bx), 'negative_outcome_boxes': int((bx.box_class == 0).sum()),
                 'enrichment_boxes': int((bx.box_class == 1).sum()), 'box_support_median': float(np.median(nB)),
                 'second_check_boxes': int(bx.eligible.sum()), 'second_check_sound': int(sound.sum()),
                 'second_check_unknown': int((bx.verified == 'unknown').sum()), 'seconds_median': P.seconds.median(),
                 'seconds_p95': P.seconds.quantile(0.95), 'checks': d['stats'].get('checks'),
                 'search_unknown': d['stats'].get('search_unknown'), 'memo_hit_rate': d['stats'].get('memo_hits', 0) / max(1, d['stats'].get('checks', 0))})
CAND = pd.DataFrame(CAND)
print(CAND.round(3).T.to_string())

## Tables for E2 to E6

In [ ]:
SECTIONS = []      # (title, source files, markdown)
f = {'pct': pct, 'pct2': lambda v: pct(v, 2), 'int': lambda v: f'{int(v):,}', 'f3': lambda v: f'{v:.3f}', 'f2': lambda v: f'{v:.2f}'}

# main table and models
def claim_text(kind, target):
    return f'negative outcome: at most {pct(target)} with the outcome' if kind == 'negative outcome' else f'enrichment: at least {pct(1 - target)} with the outcome'


def policy_text(kind, c, u, a):
    if pd.isna(c):
        return 'not run'
    word = 'certified' if c else 'not certified'
    return (f'{word} (upper bound {pct(u)}, target at most {pct(a)})' if kind == 'negative outcome'
            else f'{word} (lower bound {pct(1 - u)}, target at least {pct(1 - a)})')


mt = MAIN.assign(Claim=[claim_text(k, t) for k, t in MAIN[['kind', 'claim_target']].values],
                 Policy=[policy_text(k, c, u, a) for k, c, u, a in MAIN[['kind', 'policy_certified', 'U_sel', 'alpha_sel']].values],
                 Share=[pct_ci(*v) for v in MAIN[['certified_share', 'certified_share_lo', 'certified_share_hi']].values],
                 Outcome=[pct_ci(*v) for v in MAIN[['outcome_share', 'outcome_share_lo', 'outcome_share_hi']].values])
SECTIONS.append(('Table 1. Main results at the frozen targets (E1)', '16_table_main.csv',
                 md_table(mt[['task_name', 'family_name', 'Claim', 'Policy', 'Share', 'Outcome', 'outcome_bound_95_cp', 'mean_size', 'median_support']]
                          .rename(columns={'task_name': 'Task', 'family_name': 'Model', 'Share': 'Explained test patients with a certified claim (95% CI)',
                                           'Outcome': 'With the outcome among them (95% CI)', 'outcome_bound_95_cp': 'One-sided 95% bound',
                                           'mean_size': 'Mean features', 'median_support': 'Median calibration records'}),
                          {'One-sided 95% bound': pct, 'Mean features': f['f2'], 'Median calibration records': f['int']})
                 + '\n\nPolicy: Clopper-Pearson bound on the share with the outcome among released patients of calibration split 2 at delta_2/2 per kind '
                 '(upper bound for negative-outcome claims, lower bound for enrichment claims). Intervals: 95% percentile bootstrap over the explained '
                 'test patients (10,000 resamples). One-sided 95% bound: Clopper-Pearson on the test patients (upper for negative-outcome claims, lower for '
                 'enrichment claims). UCI Heart pools the four held-out sites.'))
mm = MAIN[MAIN.kind == 'negative outcome'][['task_name', 'family_name', 'auroc', 'auprc', 'brier', 'ece']]
SECTIONS.append(('Table 2. Underlying models on the full test split', '09b_model_metrics.csv, 09e_cardio_model_metrics.csv',
                 md_table(mm[mm.auroc.notna()].rename(columns={'task_name': 'Task', 'family_name': 'Model', 'auroc': 'AUROC', 'auprc': 'AUPRC',
                                                                'brier': 'Brier score', 'ece': 'ECE (15 equal-mass bins)'}),
                          {c: f['f3'] for c in ('AUROC', 'AUPRC', 'Brier score', 'ECE (15 equal-mass bins)')})))
us = UCI_SITES[UCI_SITES.targets_set == 'frozen']
SECTIONS.append(('Table 3. UCI Heart by held-out site (frozen heart targets)', '09e_uci_main.csv',
                 md_table(us.drop(columns=['targets_set']).rename(columns={'family_name': 'model', 'kind_name': 'kind'}),
                          {'alpha_claim': pct, 'coverage': pct, 'coverage_lo': pct, 'coverage_hi': pct, 'error_rate': pct, 'error_upper_95': pct,
                           'refuted_share': pct, 'auroc': f['f3'], 'prevalence': pct})))

# E2 validity
PRIOR_NAMES = {'naive': 'Per-box bound without prior', 'occam': 'SCAX, Occam prior', 'mix0.5': 'SCAX, mixture prior (frozen)'}
TV = V.groupby(['prior', 'delta1'], sort=False).agg(
    configurations=('reps', 'size'), repetitions=('V1_reps', 'max'), low_enough=('V1_verdict', lambda s: int((s == 'low enough').sum())),
    inconclusive=('V1_verdict', lambda s: int((s == 'inconclusive').sum())), excess=('V1_verdict', lambda s: int((s == 'excess').sum())),
    max_violation=('V1_rate', 'max'), mean_violation=('V1_rate', 'mean'), policy_excess=('V2_verdict', lambda s: int((s == 'excess').sum())),
    max_policy_violation=('V2_rate', 'max'), certification_rate=('cert_rate', 'mean'), policy_certified=('policy_certified', 'mean')).reset_index()
TV['prior'] = TV.prior.map(PRIOR_NAMES)
TV = TV.sort_values(['prior', 'delta1'])
SECTIONS.append(('Table 4. Validity on the planted-rule population (E2)', '10_validity.csv',
                 md_table(TV, {'delta1': pct, 'max_violation': f['pct'], 'mean_violation': f['pct2'], 'max_policy_violation': pct,
                               'certification_rate': pct, 'policy_certified': pct})
                 + '\n\nPer prior and delta_1: configurations (13 label settings x 3 calibration sizes x 4 values of M), verdicts of the three-way '
                 'rule for box certificates (one-sided 95% Clopper-Pearson limits, excess test corrected for 936 configurations), '
                 'and the policy check at delta_2 = 0.025.'))

# E1/E3 baselines
BS = rd('11_baselines_summary.csv')
for task in ('cdc_diabetes', 'brfss2015_heart'):
    b = BS[BS.task == task]
    SECTIONS.append((f'Table 5{"a" if task == "cdc_diabetes" else "b"}. Baselines, {TASK_NAMES[task]} (20 calibration re-draws)',
                     '11_baselines_summary.csv',
                     md_table(b[['kind', 'method', 'coverage', 'coverage_lo', 'coverage_hi', 'error_rate', 'above_target_share',
                                 'refuted_share', 'policy_certified', 'sound_share', 'mean_size']],
                              {'coverage': pct, 'coverage_lo': pct, 'coverage_hi': pct, 'error_rate': f['pct2'], 'above_target_share': pct,
                               'refuted_share': pct, 'policy_certified': pct, 'sound_share': pct, 'mean_size': f['f2']})
                     + '\n\ncoverage_lo and coverage_hi: 2.5% and 97.5% quantiles over the re-draws; above_target_share: share of released '
                     'patients whose box has a test error above its target; refuted_share: the same with the Bonferroni-corrected lower '
                     'limit above the target; policy_certified: share of re-draws.'))
SY = rd('11_synthetic_summary.csv')
SECTIONS.append(('Table 6. Baselines on the planted rule P4, where true risk is known (200 repetitions)', '11_synthetic_summary.csv',
                 md_table(SY, {'alpha_0': pct, 'alpha_1': pct, 'coverage': pct, 'coverage_lo': pct, 'coverage_hi': pct,
                               'claim_precision': f['pct'], 'precision_lo': pct, 'precision_hi': pct, 'any_false_claim': pct,
                               'true_selective_risk': f['pct2'], 'mean_size': f['f2']})))

# E3 ablations
AB = rd('12_ablations_summary.csv')
AB = AB[AB.source == 'final splits']
SECTIONS.append(('Table 7. Ablations on the frozen models, final splits (E3)', '12_ablations_summary.csv',
                 md_table(AB[['task', 'factor', 'level', 'kind', 'frozen', 'explained_share', 'explained_lo', 'explained_hi', 'test_error',
                              'policy_certified', 'U_sel']], {'explained_share': pct, 'explained_lo': pct, 'explained_hi': pct,
                                                              'test_error': f['pct2'], 'policy_certified': pct, 'U_sel': f['pct2']})))
AB2 = rd('12b_model_ablations.csv')
SECTIONS.append(('Table 8. Model factors, 500 explained test patients each (E3)', '12b_model_ablations.csv, 12b_model_variants.csv',
                 md_table(AB2[['task', 'factor', 'level', 'kind', 'coverage', 'explained_lo', 'explained_hi', 'error_rate', 'mean_size',
                               'refuted_share', 'above_target_share']],
                          {'coverage': pct, 'explained_lo': pct, 'explained_hi': pct, 'error_rate': f['pct2'], 'mean_size': f['f2'],
                           'refuted_share': pct, 'above_target_share': pct})))

# E4 shift
SH = pd.concat([rd('13_shift.csv').assign(targets_set='frozen'), rd('13b_shift.csv')], ignore_index=True)
SH = SH[(SH.setting != 'uci_main')]
SECTIONS.append(('Table 9. Shift: source certificates transported to target domains (E4)', '13_shift.csv, 13b_shift.csv',
                 md_table(SH[['setting', 'target', 'role', 'targets_set', 'kind_name', 'explained', 'coverage', 'error_rate', 'error_upper_95',
                              'refuted_share', 'above_target_share', 'certificate_refuted_share']],
                          {'coverage': pct, 'error_rate': f['pct2'], 'error_upper_95': pct, 'refuted_share': pct,
                           'above_target_share': pct, 'certificate_refuted_share': pct})))

# E5 subgroups
FS = rd('14_fairness_summary.csv')
SECTIONS.append(('Table 10. Subgroups: marginal and group-conditional SCAX, SGR (E5)', '14_fairness_summary.csv, 14_fairness.csv',
                 md_table(FS[['task', 'attribute', 'method', 'kind', 'certified_share', 'coverage_cost_vs_marginal', 'worst_group',
                              'worst_group_error_rate', 'worst_group_error_lower_95', 'worst_group_error_upper_95', 'groups_above_claim_target',
                              'groups_policy_certified', 'groups']],
                          {'certified_share': pct, 'coverage_cost_vs_marginal': pct, 'worst_group_error_rate': pct,
                           'worst_group_error_lower_95': pct, 'worst_group_error_upper_95': pct})))

# E6 runtime
RP = rd('15_runtime_production.csv')
SECTIONS.append(('Table 11. Runtime of the main results, seconds per distinct cell (E6)', '15_runtime_production.csv',
                 md_table(RP[['task', 'family', 'explained_patients', 'distinct_cells', 'trees_with_a_split', 'internal_nodes',
                              'median_seconds', 'p95_seconds', 'max_seconds', 'memo_hit_rate', 'search_unknown']],
                          {'median_seconds': f['f2'], 'p95_seconds': f['f2'], 'max_seconds': f['f2'], 'memo_hit_rate': pct})))
RS = rd('15_runtime.csv')
SECTIONS.append(('Table 12. Runtime scaling, timing study (E6)', '15_runtime.csv',
                 md_table(RS[['task', 'series', 'n_trees', 'max_depth', 'trees_with_a_split', 'internal_nodes', 'oof_auroc_training_portion',
                              'cells', 'censored_cells', 'median_seconds', 'median_is_lower_bound', 'p95_seconds', 'search_share', 'inflation_share']],
                          {'oof_auroc_training_portion': f['f3'], 'median_seconds': f['f2'], 'p95_seconds': f['f2'], 'search_share': pct,
                           'inflation_share': pct})))
if len(WEIGHTED):
    wt = WEIGHTED.assign(Share=[pct_ci(*v) for v in WEIGHTED[['certified_share_weighted', 'certified_share_weighted_lo', 'certified_share_weighted_hi']].values],
                         Outcome=[pct_ci(*v) for v in WEIGHTED[['outcome_share_weighted', 'outcome_share_weighted_lo', 'outcome_share_weighted_hi']].values])
    SECTIONS.append(('Table 14. Survey-weighted sensitivity analysis, frozen XGBoost (plan Section 6.2)', '16_weighted_sensitivity.csv',
                     md_table(wt[['task_name', 'kind', 'certified_share', 'Share', 'outcome_share', 'Outcome', 'prevalence_explained_test',
                                  'prevalence_explained_test_weighted', 'prevalence_table', 'prevalence_table_weighted']],
                              {'certified_share': pct, 'outcome_share': pct, 'prevalence_explained_test': pct, 'prevalence_explained_test_weighted': pct,
                               'prevalence_table': pct, 'prevalence_table_weighted': pct})
                     + '\n\nWeighted columns use the BRFSS 2015 final survey weight (_LLCPWT) of each explained test patient; intervals: 95% percentile '
                     'bootstrap over patients with the weights carried along. Descriptive only: the certificates are not weighted.'))
SECTIONS.append(('Table 15. Explanation cards: the box released to the most explained test patients', '16_example_cards.csv, 09_test_releases.csv, 09d_test_releases.csv',
                 md_table(CARDS.drop(columns=['task']), {'bound': pct, 'claim_target': pct, 'outcome_share_among_them': pct})))
if len(CAND):
    SECTIONS.append(('Table 13. Candidate explanations of the frozen models', '16_candidate_stats.csv',
                     md_table(CAND.drop(columns=['task']).set_index('task_name').T.reset_index().rename(columns={'index': 'statistic'}))))
for title, src, _ in SECTIONS:
    print(f'{title}   [{src}]')

## Figures (all from the results CSV files, 600 dpi)

In [ ]:
FIGS = []
plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.spines.top': False, 'axes.spines.right': False})


def save(fig, name, paper_figure, sources):
    for ext in ('png', 'pdf'):
        fig.savefig(FIG / f'{name}.{ext}', dpi=600, bbox_inches='tight')
    FIGS.append({'paper_figure': paper_figure, 'file': f'figures/{name}.png', 'sources': sources})
    plt.show()
    plt.close(fig)


def pct_axis(ax, axis='y', signed=False):
    """Percent tick labels with as few decimals as the tick positions need (no rounding of a tick label)."""
    a = ax.yaxis if axis == 'y' else ax.xaxis
    locs = list(a.get_majorticklocs())
    dec = next(d for d in range(4) if all(abs(round(100 * v, d) - 100 * v) < 1e-7 for v in locs) or d == 3)
    a.set_major_locator(FixedLocator(locs))
    a.set_major_formatter(FuncFormatter(lambda v, _: (f'{100 * v:+.{dec}f}%' if signed and abs(v) > 1e-12 else f'{100 * v:.{dec}f}%').replace('-', '−')))

In [ ]:
# Figures 1 and 2 of the manuscript: framework and mechanism (schematic, no data)
from matplotlib.patches import FancyBboxPatch, Rectangle


def figure_framework(save):
    plt.rcParams.update({'font.family': 'DejaVu Sans'})
    fig = plt.figure(figsize=(7.4, 4.3))
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 60); ax.axis('off')
    cols = [('Data', '#2F6FB5', '#E7F0FA'), ('Model', '#D9622B', '#FCEBE2'), ('Formal explainer', '#1E9E74', '#E2F4EC'),
            ('Box certificates', '#C98A00', '#FBF1D9'), ('Policy certificate', '#C2457D', '#F9E6EF'), ('Output', '#2E7D32', '#E6F2E6')]
    steps = ['Step 1', 'Step 2', 'Step 3', 'Step 4', 'Step 5', 'Step 6']
    boxes = [
        ['Training portion, 40%\n(dev-train and\ndev-cal)', 'Calibration split 1, 30%', 'Calibration split 2, 15%', 'Test split, 15%'],
        ['Tree ensemble\n(XGBoost, LightGBM\nor random forest)', 'Exported classifier:\nexact arithmetic,\nthreshold $\\tau$', 'Agreement with the\nnative library checked'],
        ['Threshold grid: a finite\nfamily of boxes', 'Candidate boxes: AXps\nalong $M$ feature orders,\nevery sound box on\nthe inflation path', 'Soundness: exact\nsearch; a second exact\ncheck re-proves each\nreleasable box'],
        ['Prior $\\pi(B)$ over boxes\n(training portion only)', 'Clopper-Pearson bound\n$\\hat U(B)$ at level\n$\\delta_1\\pi(B)$', 'Release the shortest\nbox with\n$\\hat U(B)\\leq\\alpha_{\\hat y}$'],
        ['Frozen release policy\napplied to split 2', 'Bound $\\hat U_{\\mathrm{sel}}$ on the\nerror among released\npatients ($\\delta_2/2$\nper claim kind)', 'Deployed only if\n$\\hat U_{\\mathrm{sel}}\\leq\\alpha_{\\mathrm{sel}}$'],
        ['Prediction + proved\nexplanation + certified\nclaim on the recorded\noutcome', 'Abstain: no prediction\nis shown', '"Policy not certified":\nnot deployed'],
    ]
    uses = ['label-blind random\nallocation', 'uses: training\nportion', 'uses: training\nportion', 'uses: training (prior)\nand split 1 (bounds)',
            'uses: split 2,\nonce', 'reported on the\ntest split']
    n, gap, left = 6, 1.2, 1.0
    w = (100 - 2 * left - gap * (n - 1)) / n
    top, bottom = 59.5, 10.5
    centres = []
    for i, ((title, dark, light), bx) in enumerate(zip(cols, boxes)):
        x0 = left + i * (w + gap)
        ax.add_patch(Rectangle((x0, bottom), w, top - bottom, facecolor=light, edgecolor='none', zorder=0))
        ax.add_patch(Rectangle((x0, top - 4.2), w, 4.2, facecolor=dark, edgecolor='none', zorder=1))
        ax.text(x0 + w / 2, top - 2.1, title, ha='center', va='center', color='white', fontsize=7.6, fontweight='bold', zorder=2)
        ax.text(x0 + w / 2, top - 6.0, steps[i], ha='center', va='center', color='0.35', fontsize=6.2)
        ys = np.linspace(47.5, 24.5, len(bx)) if len(bx) > 3 else [47.5, 36.0, 24.5]
        hs = [7.6] * len(bx) if i != 0 else [7.6, 5.0, 5.0, 5.0]
        if i == 0:
            ys = [47.5, 38.0, 31.0, 24.0]
        cy = []
        for j, (txt, y, h) in enumerate(zip(bx, ys, hs)):
            ec = dark
            if i == 0 and j > 0:
                ec = [cols[3][1], cols[4][1], cols[5][1]][j - 1]
            if i == 5 and j > 0:
                ec = '0.45'
            ax.add_patch(FancyBboxPatch((x0 + 0.8, y - h / 2), w - 1.6, h, boxstyle='round,pad=0.25,rounding_size=0.9',
                                        facecolor='white', edgecolor=ec, linewidth=1.1, zorder=2))
            ax.text(x0 + w / 2, y, txt, ha='center', va='center', fontsize=6.0, zorder=3, linespacing=1.25)
            cy.append((y, h))
        if i > 0:
            for (y1, h1), (y2, h2) in zip(cy[:-1], cy[1:]):
                ax.annotate('', xy=(x0 + w / 2, y2 + h2 / 2 + 0.35), xytext=(x0 + w / 2, y1 - h1 / 2 - 0.35),
                            arrowprops=dict(arrowstyle='-|>', color='0.35', lw=0.9, mutation_scale=7), zorder=3)
        ax.add_patch(FancyBboxPatch((x0 + 1.6, bottom + 1.3), w - 3.2, 4.2, boxstyle='round,pad=0.2,rounding_size=0.8',
                                    facecolor='none', edgecolor=dark, linewidth=0.9, linestyle=(0, (3, 2)), zorder=2))
        ax.text(x0 + w / 2, bottom + 3.4, uses[i], ha='center', va='center', fontsize=5.6, color='0.2', zorder=3)
        centres.append((x0, cy[0]))
    for i in range(n - 1):
        x0, (y, h) = centres[i]
        x1 = centres[i + 1][0]
        ax.annotate('', xy=(x1 + 0.7, 47.5), xytext=(x0 + w - 0.7, 47.5),
                    arrowprops=dict(arrowstyle='-|>', color='black', lw=1.2, mutation_scale=9), zorder=4)
    ax.add_patch(FancyBboxPatch((2.0, 1.0), 96, 6.6, boxstyle='round,pad=0.3,rounding_size=1.2', facecolor='#F3F2EE',
                                edgecolor='0.35', linewidth=1.0))
    ax.text(50, 4.3, 'Frozen on the training portion and recorded with a hash before calibration split 1 is opened:\n'
            'model, threshold $\\tau$, box family, prior $\\pi$, number of feature orders $M$, release rule, targets '
            '$\\alpha_0$, $\\beta$, $\\alpha_{\\mathrm{sel}}$; failure budget $\\delta_1+\\delta_2=0.05$',
            ha='center', va='center', fontsize=6.6, linespacing=1.45)
    save(fig)


def figure_mechanism(save):
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7.5})
    rng = np.random.default_rng(7)
    fig = plt.figure(figsize=(7.4, 3.3))
    ax = fig.add_axes([0.07, 0.15, 0.37, 0.72])
    n = 1150
    age = rng.integers(1, 14, n) + rng.uniform(-0.32, 0.32, n)
    bmi = np.clip(rng.normal(28, 5.5, n), 15.5, 49.5)
    p = 1 / (1 + np.exp(-(-6.2 + 0.42 * np.round(age) + 0.06 * (bmi - 28))))
    y = rng.random(n) < p
    ax.scatter(age[~y], bmi[~y], s=3.2, color='#2F6FB5', alpha=0.55, lw=0, zorder=1)
    ax.scatter(age[y], bmi[y], s=4.2, color='#D9622B', alpha=0.9, lw=0, zorder=2)
    ax.add_patch(Rectangle((0.5, 15), 4.0, 12, fill=False, ec='black', lw=1.5, zorder=3))
    ax.add_patch(Rectangle((0.5, 15), 6.0, 17, fill=False, ec='#1E9E74', lw=1.8, zorder=3))
    ax.add_patch(Rectangle((0.5, 15), 8.0, 22, fill=False, ec='#C2457D', lw=1.5, ls=(0, (4, 2)), zorder=3))
    ax.plot(3, 24, marker='*', ms=11, color='black', zorder=4)
    ax.annotate('patient', xy=(3, 24.3), xytext=(1.2, 46), fontsize=7, arrowprops=dict(arrowstyle='-', color='0.3', lw=0.7))
    ax.set_xlim(0.4, 13.6); ax.set_ylim(15, 50)
    ax.set_xticks(range(1, 14))
    ax.set_xlabel('Age group (1 = 18 to 24, ..., 13 = 80 or older)')
    ax.set_ylabel('Body mass index (kg/m$^2$)')
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.set_title('(a) Each sound box gets its own certificate\n(illustration with simulated points; target 5%)', fontsize=7.8, loc='left')
    lx = fig.add_axes([0.46, 0.15, 0.19, 0.72]); lx.axis('off'); lx.set_xlim(0, 1); lx.set_ylim(0, 1)
    items = [('black', '-', 'AXp box', '$n_B$ = 420, errors = 4\nbound 5.5% > 5%:\nnot certified'),
             ('#1E9E74', '-', 'Inflated box (path step)', '$n_B$ = 3,500, errors = 60\nbound 3.0% $\\leq$ 5%:\ncertified and released'),
             ('#C2457D', (0, (4, 2)), 'Widest inflated box', '$n_B$ = 6,100, errors = 420\nbound 8.5% > 5%:\nnot certified')]
    for k, (c, ls, head, body) in enumerate(items):
        yy = 0.95 - k * 0.28
        lx.plot([0, 0.12], [yy, yy], color=c, ls=ls, lw=1.8)
        lx.text(0.16, yy, head, va='center', fontsize=7, fontweight='bold')
        lx.text(0.16, yy - 0.04, body, va='top', fontsize=6.3, color='0.25', linespacing=1.3)
    lx.scatter([0.05], [0.12], s=14, color='#2F6FB5'); lx.text(0.16, 0.12, 'Recorded outcome: no', va='center', fontsize=6.5)
    lx.scatter([0.05], [0.04], s=14, color='#D9622B'); lx.text(0.16, 0.04, 'Recorded outcome: yes', va='center', fontsize=6.5)
    bx = fig.add_axes([0.71, 0.15, 0.29, 0.72]); bx.axis('off'); bx.set_xlim(0, 10); bx.set_ylim(0, 10)
    bx.text(0, 9.75, '(b) Why the release policy needs\nits own test', fontsize=7.8, va='top')
    bx.add_patch(Rectangle((0.3, 3.6), 5.4, 4.0, facecolor='#E7F0FA', edgecolor='#2F6FB5', lw=1.6))
    bx.add_patch(Rectangle((3.6, 3.6), 5.9, 4.0, facecolor='#FCEBE2', edgecolor='#D9622B', lw=1.6, alpha=0.75))
    bx.text(1.95, 5.6, '5 errors', ha='center', va='center', fontsize=7)
    bx.text(4.65, 5.6, '95\ncorrect', ha='center', va='center', fontsize=7.4, fontweight='bold')
    bx.text(7.55, 5.6, '5 errors', ha='center', va='center', fontsize=7)
    bx.text(0.3, 7.95, 'Box 1: 5 / 100 = 5%', color='#2F6FB5', fontsize=7, fontweight='bold')
    bx.text(9.5, 3.0, 'Box 2: 5 / 100 = 5%', color='#D9622B', fontsize=7, fontweight='bold', ha='right')
    bx.text(4.9, 1.75, 'Released together: 10 / 105 = 9.5%', ha='center', fontsize=7.4, fontweight='bold')
    bx.text(4.9, 0.95, 'Valid box certificates do not bound the error among\nreleased patients; the policy is certified on split 2.',
            ha='center', va='top', fontsize=6.2, color='0.3')
    save(fig)


figure_framework(lambda fig: save(fig, 'fig_framework', 'Figure 1', 'schematic (no data)'))
figure_mechanism(lambda fig: save(fig, 'fig_mechanism', 'Figure 2', 'schematic (simulated points, illustrative numbers)'))
plt.rcParams.update({'font.size': 9})

In [ ]:
# Figure 3 and Figure 5: naive certification against SCAX; the overlap problem (notebook 10)
V10 = V                                        # 10_validity.csv, with notebook 10b's repetitions where present
MIX = [p for p in V10.prior.unique() if p.startswith('mix')][0]
N1_LIST, M_LIST, DELTA2 = sorted(V10.n1.unique()), sorted(V10.M.unique()), 0.025
BLUE, VERMILION, GREEN = '#0072B2', '#D55E00', '#009E73'
SERIES = (('naive', VERMILION, 'o', 'Per-box bound without prior'), ('occam', BLUE, 's', 'SCAX, Occam prior'),
          (MIX, GREEN, '^', 'SCAX, mixture prior'))
plt.rcParams.update({'font.size': 9})
sub = V10[(V10.delta1 == 0.025) & ((V10.eps > 0) | (V10.rule == 'P4'))]
fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.9), sharey=True)
for ax, n1 in zip(axes, N1_LIST):
    s = sub[sub.n1 == n1].groupby(['prior', 'M']).V1_rate.mean().unstack(0)
    for prior, col, mk, lab in SERIES:
        ax.plot(s.index, s[prior], marker=mk, color=col, lw=2, ms=5, label=lab)
    ax.axhline(0.025, color='0.4', lw=1, ls='--')
    ax.set_title(f'{n1:,} calibration records', fontsize=9)
    ax.set_xlabel('Feature orders per patient (M)')
    ax.set_xticks(M_LIST); ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel('Share of repetitions with\na box above its bound')
axes[0].text(max(M_LIST), 0.06, 'target 0.025', ha='right', fontsize=7, color='0.3')
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.1, 1, 1))
save(fig, 'fig_validity_naive_vs_scax', 'Figure 3', '10_validity.csv')

p4 = V10[(V10.rule == 'P4') & (V10.delta1 == 0.025) & (V10.n1 == max(N1_LIST))]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharey=True)
for prior, col, mk, lab in SERIES:
    s = p4[p4.prior == prior].sort_values('M')
    axes[0].plot(s.M, s.overlap / s.reps, marker=mk, color=col, lw=2, ms=5, label=lab)
    axes[1].plot(s.M, s.V2 / s.reps, marker=mk, color=col, lw=2, ms=5, label=lab)
axes[0].set_title('Every released box within target,\nreleased patients above target', fontsize=9)
axes[1].set_title('Policy certified while released\npatients are above target', fontsize=9)
axes[1].axhline(DELTA2, color='0.4', lw=1, ls='--')
axes[1].text(max(M_LIST), DELTA2 + 0.03, 'target 0.025', ha='right', fontsize=7, color='0.3')
for ax in axes:
    ax.set_xlabel('Feature orders per patient (M)'); ax.set_xticks(M_LIST); ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel('Share of repetitions')
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.1, 1, 1))
save(fig, 'fig_validity_overlap_p4', 'Figure 5', '10_validity.csv')

# Figure 4: violation frequency against delta_1 (with the verdicts) and the certification rate it costs
plt.rcParams.update({'font.size': 8})
D1 = sorted(V.delta1.unique())
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.2), gridspec_kw={'width_ratios': [1.25, 1]})
ax = axes[0]
rng = np.random.default_rng(SEED)
for j, (prior, col, mk, lab) in enumerate(SERIES[1:]):
    for i, d1 in enumerate(D1):
        s = V[(V.prior == prior) & (V.delta1 == d1)]
        x = i + (j - 0.5) * 0.36 + rng.uniform(-0.12, 0.12, len(s))
        ax.scatter(x, s.V1_rate, s=7, color=col, alpha=0.45, lw=0, marker=mk, label=lab if i == 0 else None)
        ax.plot([i + (j - 0.5) * 0.36 - 0.15, i + (j - 0.5) * 0.36 + 0.15], [s.V1_upper.max()] * 2, color=col, lw=1.2)
        verdicts = s.V1_verdict.value_counts()
        ax.text(i + (j - 0.5) * 0.36, -0.0065, f"{verdicts.get('inconclusive', 0)}/{verdicts.get('excess', 0)}", ha='center',
                va='top', fontsize=6, color=col)
    for i, d1 in enumerate(D1):
        ax.plot([i - 0.45, i + 0.45], [d1, d1], color='0.25', lw=1, ls='--', zorder=1)
ax.set_xticks(range(len(D1))); ax.set_xticklabels([f'{100 * d:g}%' for d in D1])
ax.set_xlabel(r'Failure budget for box certificates ($\delta_1$)')
ax.set_ylabel('Share of repetitions with a box above its bound')
top = max(0.06, float(V[V.prior != 'naive'].V1_upper.max()) * 1.1)
ax.set_ylim(-0.012, top)
ax.set_yticks(np.arange(0, top + 1e-9, 0.01)); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_title('Box certificates: violation frequency', fontsize=8.5)
ax.text(0.01, 0.99, 'dots: configurations; bar: largest upper 95% limit;\ndashed: target; numbers: inconclusive/excess',
        transform=ax.transAxes, va='top', fontsize=6.3, color='0.3')
ax = axes[1]
RULE_COL = dict(zip(['P1', 'P2', 'P3', 'P4'], ['#0072B2', '#E69F00', '#009E73', '#CC79A7']))
for rule, col in RULE_COL.items():
    s = V[(V.prior == MIX) & (V.rule == rule)].groupby('delta1').cert_rate.mean()
    ax.plot(range(len(s)), s.values, marker='o', ms=4, lw=1.4, color=col, label=f'{rule}' + (' (noise 0 to 0.20)' if rule != 'P4' else ''))
ax.set_xticks(range(len(D1))); ax.set_xticklabels([f'{100 * d:g}%' for d in D1])
ax.set_xlabel(r'Failure budget for box certificates ($\delta_1$)')
ax.set_ylabel('Share of the population with a certified claim')
ax.set_ylim(0, 1.02); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_title('SCAX, mixture prior: certification rate', fontsize=8.5)
ax.legend(frameon=False, fontsize=6.8, loc='lower right')
for a in axes:
    a.grid(axis='y', color='0.92', lw=0.6); a.set_axisbelow(True)
h = [Line2D([], [], marker=mk, ls='none', color=col, ms=5, label=lab) for _, col, mk, lab in SERIES[1:]]
fig.legend(handles=h, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.32, 1.04), fontsize=7.5)
fig.tight_layout(rect=(0, 0, 1, 0.94))
save(fig, 'fig_validity_delta', 'Figure 4', '10_validity.csv' + (', 10b_validity_more_repetitions.csv' if MORE.exists() else ''))

In [ ]:
# Figure 5: certification-rate curves (three tasks, three families); Figures S2 to S5 as in notebooks 09, 09b, 09d, 09e
STYLE_F = {'xgboost': ('#0072B2', 'o'), 'lightgbm': ('#D55E00', 's'), 'sklearn_rf': ('#009E73', '^')}
A09B, A09E = rd('09b_main.csv'), rd('09e_cardio_main.csv')
plt.rcParams.update({'font.size': 7.5})
fig, axes = plt.subplots(3, 2, figsize=(7.4, 7.4))
for i, task in enumerate(('brfss2015_heart', 'cdc_diabetes', 'cardio70k')):
    T = TARGETS[task]
    for fmy, (col, mk) in STYLE_F.items():
        if task == 'cardio70k':
            d = A09E[A09E.family == fmy]
            neg = d[d.kind == 'negative outcome'].drop_duplicates('alpha_0').sort_values('alpha_0')
            enr = d[d.kind == 'enrichment'].drop_duplicates('beta').sort_values('beta')
        else:
            d = A09B[(A09B.task == task) & (A09B.family == fmy)]
            neg = d[(d.kind == 'negative outcome') & (d.beta == T['beta'])].sort_values('alpha_0')
            enr = d[(d.kind == 'enrichment') & (d.alpha_0 == T['alpha_0'])].sort_values('beta')
        for ax, e, x in ((axes[i, 0], neg, 'alpha_0'), (axes[i, 1], enr, 'beta')):
            ax.plot(e[x], e.test_certified_share, color=col, lw=1.3, zorder=2, label=FAMILY_NAMES[fmy])
            plan = e.grid.eq('plan').to_numpy() if 'grid' in e else np.zeros(len(e), bool)
            ax.plot(e[x][~plan], e.test_certified_share[~plan], mk, ms=3.6, color=col, zorder=3)
            ax.plot(e[x][plan], e.test_certified_share[plan], mk, ms=3.6, mfc='white', mec=col, mew=1, zorder=3)
    for ax, x0 in ((axes[i, 0], T['alpha_0']), (axes[i, 1], T['beta'])):
        ax.axvline(x0, color='0.3', ls='--', lw=0.8, zorder=1)
        ax.set_ylim(bottom=0); ax.grid(color='0.92', lw=0.6); ax.set_axisbelow(True)
        ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    axes[i, 0].set_ylabel(f'{TASK_NAMES[task]}:\nshare of explained test patients')
axes[0, 0].set_title('Certified negative-outcome explanations', fontsize=8.5)
axes[0, 1].set_title('Certified enrichment explanations', fontsize=8.5)
axes[2, 0].set_xlabel(r'Target error for negative-outcome claims ($\alpha_0$)')
axes[2, 1].set_xlabel(r'Claimed minimum share with the outcome ($\beta$)')
h, l = axes[0, 0].get_legend_handles_labels()
h += [Line2D([], [], color='0.3', ls='--', lw=0.8), Line2D([], [], marker='o', ls='none', mfc='white', mec='0.3', ms=4)]
l += ['Frozen target (policy test)', "Cardio-70k: plan's grid (descriptive)"]
fig.legend(h, l, loc='upper center', ncol=5, frameon=False, fontsize=7, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.965))
save(fig, 'fig_certified_share_curves', 'Figure 6', '09b_main.csv, 09e_cardio_main.csv')

A09 = rd('09_main.csv')
plt.rcParams.update({'font.size': 9})
NAMES2 = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease'}
COLORS2 = {'cdc_diabetes': '#0072B2', 'brfss2015_heart': '#D55E00'}
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))
for task in ('cdc_diabetes', 'brfss2015_heart'):
    T = TARGETS[task]
    neg = A09[(A09.task == task) & (A09.kind == 'negative outcome') & (A09.beta == T['beta'])].sort_values('alpha_0')
    enr = A09[(A09.task == task) & (A09.kind == 'enrichment') & (A09.alpha_0 == T['alpha_0'])].sort_values('beta')
    axes[0].plot(neg.alpha_0, neg.test_certified_share, marker='o', lw=2, color=COLORS2[task], label=NAMES2[task])
    axes[1].plot(enr.beta, enr.test_certified_share, marker='o', lw=2, color=COLORS2[task], label=NAMES2[task])
axes[0].set_xlabel('Target error for negative-outcome claims'); axes[0].set_ylabel('Share of test patients')
axes[0].set_title('Certified negative-outcome explanations', fontsize=9)
axes[1].set_xlabel('Claimed minimum share with the outcome'); axes[1].set_title('Certified enrichment explanations', fontsize=9)
for ax in axes:
    ax.set_ylim(0, 1)
axes[0].legend(frameon=False, loc='upper left')
fig.tight_layout()
save(fig, 'fig_certified_share_vs_target', 'not in the manuscript (covered by Figure 6)', '09_main.csv')

plt.rcParams.update({'font.size': 8})
fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.2))
for i, task in enumerate(('cdc_diabetes', 'brfss2015_heart')):
    T = TARGETS[task]
    for fmy, (col, mk) in STYLE_F.items():
        d = A09B[(A09B.task == task) & (A09B.family == fmy)]
        neg = d[(d.kind == 'negative outcome') & (d.beta == T['beta'])].sort_values('alpha_0')
        enr = d[(d.kind == 'enrichment') & (d.alpha_0 == T['alpha_0'])].sort_values('beta')
        axes[i, 0].plot(neg.alpha_0, neg.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.4, label=FAMILY_NAMES[fmy])
        axes[i, 1].plot(enr.beta, enr.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.4, label=FAMILY_NAMES[fmy])
    axes[i, 0].set_ylabel(f'{NAMES2[task]}:\nshare of test patients')
    for ax in axes[i]:
        ax.set_ylim(bottom=0); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.grid(color='0.92', lw=0.6)
        ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
axes[0, 0].set_title('Certified negative-outcome explanations', fontsize=8.5)
axes[0, 1].set_title('Certified enrichment explanations', fontsize=8.5)
axes[1, 0].set_xlabel(r'Target error for negative-outcome claims ($\alpha_0$)')
axes[1, 1].set_xlabel(r'Claimed minimum share with the outcome ($\beta$)')
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=3, frameon=False, fontsize=7.5, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, 'fig_certified_share_by_family', 'not in the manuscript (covered by Figure 6)', '09b_main.csv')

A09D = rd('09d_main.csv'); T = TARGETS['cardio70k']
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0))
neg = A09D[A09D.kind == 'negative outcome'].drop_duplicates('alpha_0').sort_values('alpha_0')
enr = A09D[A09D.kind == 'enrichment'].drop_duplicates('beta').sort_values('beta')
for ax, d, x, scaled, lab, main in ((axes[0], neg, 'alpha_0', T['alpha_0_grid'], r'Target error for negative-outcome claims ($\alpha_0$)', T['alpha_0']),
                                    (axes[1], enr, 'beta', T['beta_grid'], r'Claimed minimum share with the outcome ($\beta$)', T['beta'])):
    on = d[x].isin(scaled)
    ax.plot(d[x], d.test_certified_share, color='#0072B2', lw=1.4, zorder=2)
    ax.plot(d[x][on], d.test_certified_share[on], 'o', ms=5, color='#0072B2', label='Scaled grid', zorder=3)
    ax.plot(d[x][~on], d.test_certified_share[~on], 'o', ms=5, mfc='white', mec='#0072B2', mew=1.2, label="Plan's grid (descriptive)", zorder=3)
    m = d[np.isclose(d[x], main)]
    ax.plot(m[x], m.test_certified_share, 'o', ms=10, mfc='none', mec='#D55E00', mew=1.5, label='Frozen target (policy test)', zorder=4)
    ax.set_xlabel(lab)
    ax.set_ylim(0, max(0.05, 1.15 * float(d.test_certified_share.max())))
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.grid(color='0.92', lw=0.6); ax.set_axisbelow(True)
axes[0].set_ylabel('Cardio-70k: share of test patients')
axes[0].set_title('Certified negative-outcome explanations', fontsize=8.5)
axes[1].set_title('Certified enrichment explanations', fontsize=8.5)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, fontsize=7, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.07, 1, 1))
save(fig, 'fig_cardio_certified_share', 'not in the manuscript (covered by Figure 6)', '09d_main.csv')

plt.rcParams.update({'font.size': 7.5})
ALPHA_H = {0: TARGETS['brfss2015_heart']['alpha_0'], 1: TARGETS['brfss2015_heart']['alpha_1']}
fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.6))
for fmy, (col, mk) in STYLE_F.items():
    d = A09E[A09E.family == fmy]
    neg = d[d.kind == 'negative outcome'].drop_duplicates('alpha_0').sort_values('alpha_0')
    enr = d[d.kind == 'enrichment'].drop_duplicates('beta').sort_values('beta')
    axes[0, 0].plot(neg.alpha_0, neg.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.3, label=FAMILY_NAMES[fmy])
    axes[0, 1].plot(enr.beta, enr.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.3)
for ax, x0 in ((axes[0, 0], T['alpha_0']), (axes[0, 1], T['beta'])):
    ax.axvline(x0, color='0.3', ls='--', lw=0.8)
    ax.set_ylim(bottom=0); ax.grid(color='0.92', lw=0.6); ax.set_axisbelow(True)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
axes[0, 0].set_xlabel(r'Target error for negative-outcome claims ($\alpha_0$)')
axes[0, 1].set_xlabel(r'Claimed minimum share with the outcome ($\beta$)')
axes[0, 0].set_ylabel('Cardio-70k: share of test patients')
axes[0, 0].set_title('Certified negative-outcome explanations', fontsize=8.5)
axes[0, 1].set_title('Certified enrichment explanations', fontsize=8.5)
sites = [uci.SITE_NAMES[s] for s in uci.SITES]
w = 0.26
for j, (fmy, (col, mk)) in enumerate(STYLE_F.items()):
    d = U[(U.role == 'target') & (U.kind == 1) & (U.family == fmy) & (U.targets_set == 'frozen')]
    d = d.set_index(d.target.str.replace(' held out', '', regex=False)).reindex(sites)
    xs = np.arange(len(sites)) + (j - 1) * w
    axes[1, 0].bar(xs, d.coverage, width=w, color=col, edgecolor='white', linewidth=0.5)
    ok = d.accepted.fillna(0).to_numpy() > 0
    prec, lo = 1 - d.error_rate.to_numpy(), 1 - d.error_upper_95.to_numpy()
    axes[1, 1].errorbar(xs[ok], prec[ok], yerr=[prec[ok] - lo[ok], np.zeros(ok.sum())], fmt=mk, ms=4, color=col, elinewidth=0.8,
                        capsize=2.5, capthick=0.8)
axes[1, 1].axhline(1 - ALPHA_H[1], color='0.3', ls='--', lw=0.8)
for ax in axes[1]:
    ax.set_xticks(np.arange(len(sites))); ax.set_xticklabels([s.replace('VA Long Beach', 'VA Long\nBeach') for s in sites])
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.grid(axis='y', color='0.92', lw=0.6); ax.set_axisbelow(True)
    ax.set_xlabel('UCI Heart: site held out')
axes[1, 0].set_ylabel('Share of held-out patients with a\ncertified enrichment explanation')
axes[1, 1].set_ylabel('Share with angiographic disease among them\n(one-sided 95% lower bound)')
axes[1, 1].set_ylim(0, 1.02)
axes[1, 0].set_title(r'UCI Heart, enrichment claims ($\beta$ = ' + f'{1 - ALPHA_H[1]:.0%})', fontsize=8.5)
axes[1, 1].set_title('UCI Heart, claim checked on the held-out site', fontsize=8.5)
h, l = axes[0, 0].get_legend_handles_labels()
h.append(Line2D([], [], color='0.3', ls='--', lw=0.8)); l.append('Frozen target / claimed minimum share')
fig.legend(h, l, loc='upper center', ncol=4, frameon=False, fontsize=7.5, bbox_to_anchor=(0.5, 1.01))
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, 'fig_families_cardio_uci', 'Figure A1', '09e_cardio_main.csv, 09e_uci_main.csv')

In [ ]:
# Figure 6 and Figure S6: baselines (notebook 11)
STYLE_B = {'SCAX': ('#0072B2', 'o'), 'Per-box bound without prior': ('#D55E00', 's'), 'Empirical error rule': ('#CC79A7', 'D'),
           'Uncertified AXp': ('#E69F00', '^'), 'Uncertified inflated box': ('#56B4E9', 'v'),
           'Rule dictionary': ('#009E73', 'P'), 'Rule dictionary, sound rules only': ('#009E73', 'X'),
           'Confidence-aware AXp': ('#000000', '*')}
plt.rcParams.update({'font.size': 8})
SYN_TARGETS = SY[['alpha_0', 'alpha_1']].drop_duplicates().values.tolist()
fig, axes = plt.subplots(1, len(SYN_TARGETS), figsize=(7.4, 3.3), sharey=True)
for ax, (a0, a1) in zip(np.atleast_1d(axes), SYN_TARGETS):
    s = SY[(SY.alpha_0 == a0) & (SY.alpha_1 == a1)]
    for _, r in s.iterrows():
        col, mk = STYLE_B[r.method]
        if pd.isna(r.claim_precision):
            continue
        ax.errorbar(r.coverage, r.claim_precision, xerr=[[r.coverage - r.coverage_lo], [r.coverage_hi - r.coverage]],
                    yerr=[[r.claim_precision - r.precision_lo], [r.precision_hi - r.claim_precision]],
                    fmt=mk, color=col, ms=7 if r.method == 'SCAX' else 5, mfc=col if 'sound' not in r.method else 'white',
                    lw=1, capsize=0, label=r.method)
    ax.set_title(f'Targets: {a0:.0%} for negative-outcome claims,\n{a1:.0%} for enrichment claims', fontsize=8.5)
    ax.set_xlabel('Share of the population given an explanation'); ax.set_xlim(-0.02, 1.02)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
np.atleast_1d(axes)[0].set_ylabel('Share of released explanations\nwhose claim is true')
hl = {}
for ax in np.atleast_1d(axes):
    for h_, l_ in zip(*ax.get_legend_handles_labels()):
        hl.setdefault(l_, h_)
fig.legend(list(hl.values()), list(hl.keys()), loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.01), fontsize=7.5)
fig.tight_layout(rect=(0, 0.16, 1, 1))
save(fig, 'fig_coverage_vs_true_claim_precision', 'Figure 7', '11_synthetic_summary.csv')

SHOW = ['SCAX', 'Per-box bound without prior', 'Empirical error rule', 'Uncertified AXp, all patients',
        'Uncertified AXp, matched coverage', 'Uncertified inflated box, all patients', 'Rule dictionary']
LABEL = {'Uncertified AXp, all patients': 'Uncertified AXp (all)', 'Uncertified AXp, matched coverage': 'Uncertified AXp (SCAX coverage)',
         'Uncertified inflated box, all patients': 'Uncertified inflated box (all)'}
fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.0), sharex=True)
for i, task in enumerate(('cdc_diabetes', 'brfss2015_heart')):
    for j, kind in enumerate(['negative outcome', 'enrichment']):
        ax = axes[i, j]
        s = BS[(BS.task == task) & (BS.kind == kind)].set_index('method').reindex(SHOW)
        yv = np.arange(len(SHOW))[::-1]
        ax.barh(yv, s.above_target_share.fillna(0), color=['#0072B2' if m == 'SCAX' else '#999999' for m in SHOW], height=0.6)
        ax.set_yticks(yv); ax.set_yticklabels([LABEL.get(m, m) for m in SHOW] if j == 0 else [])
        ax.set_title(f'{TASK_NAMES[task]}, {kind} claims', fontsize=8.5)
        for m, y0, v, c, snd in zip(SHOW, yv, s.above_target_share.fillna(0), s.coverage, s.sound_share):
            extra = f'; sound for the model: {snd:.0%}' if m == 'Rule dictionary' and snd == snd else ''
            ax.text(v + 0.01, y0, f'{v:.0%}   (explained: {c:.0%}{extra})', va='center', fontsize=6.5)
        ax.set_xlim(0, 1); ax.set_xticks([0, 0.25, 0.5, 0.75]); ax.xaxis.set_major_formatter(PercentFormatter(1.0))
fig.supxlabel('Share of released explanations whose box has test error above its target', fontsize=8.5)
fig.tight_layout()
save(fig, 'fig_baselines_claims_real', 'Figure A2', '11_baselines_summary.csv')

In [ ]:
# Figure 7, Figures S7 and S8: shift (notebooks 13 and 13b)
REC13, TGT13, TRN13, BOX13 = rd('13_shift_recertification.csv'), rd('13_shift_targets.csv'), rd('13_shift.csv'), rd('13_shift_boxes.csv')
SIZES13, DRAWS13 = rj('13_shift.json')['sizes']['recertification_sizes'], rj('13_shift.json')['sizes']['draws']
SIZE_LABEL = {500: '500', 1000: '1k', 2000: '2k', 5000: '5k', 10000: '10k'}
OKABE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00']
PANELS = [('temporal', 'Temporal: 2015 model, later years'), ('geographic', 'Census region held out'), ('state', 'States with most records')]
plt.rcParams.update({'font.size': 8})
fig = plt.figure(figsize=(7.4, 6.1))
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.36], hspace=0.24, wspace=0.3, left=0.1, right=0.98, top=0.95, bottom=0.035)
axes = np.empty((2, 3), dtype=object)
for j in range(3):
    axes[0, j] = fig.add_subplot(gs[0, j])
    axes[1, j] = fig.add_subplot(gs[1, j], sharex=axes[0, j])
for j, (setting, title) in enumerate(PANELS):
    tg = TGT13[(TGT13.setting == setting) & (TGT13.role == 'target')].target.tolist()
    ref_setting = 'temporal' if setting == 'temporal' else 'geographic'
    for i, k in enumerate((0, 1)):
        ax = axes[i, j]
        rf = REC13[(REC13.setting == ref_setting) & (REC13.role == 'reference') & (REC13.kind == k)].groupby('m').coverage.mean()
        ax.plot(rf.index, rf.values, color='0.6', lw=1.6, marker='s', ms=2.5,
                label='No shift (source test split)' if setting == 'temporal' else 'No shift (mean of the 4 models)')
        for c, t in enumerate(tg):
            col = OKABE[c % len(OKABE)]
            s = REC13[(REC13.setting == setting) & (REC13.target == t) & (REC13.kind == k)].sort_values('m')
            ax.fill_between(s.m, s.coverage_q025, s.coverage_q975, color=col, alpha=0.13, lw=0)
            ax.plot(s.m, s.coverage, color=col, lw=1.2, marker='o', ms=2.8, label=t)
            tr = TRN13[(TRN13.setting == setting) & (TRN13.target == t) & (TRN13.kind == k)].coverage
            if len(tr):
                ax.axhline(float(tr.iloc[0]), color=col, lw=0.9, ls=(0, (4, 2)))
        ax.set_xscale('log')
        ax.xaxis.set_major_locator(FixedLocator(SIZES13))
        ax.xaxis.set_major_formatter(FixedFormatter([SIZE_LABEL.get(m, str(m)) for m in SIZES13]))
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set_ylim(bottom=0)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.grid(color='0.92', lw=0.6)
        if i == 0:
            ax.set_title(title, fontsize=8.5)
            ax.tick_params(labelbottom=False)
        if j == 0:
            ax.set_ylabel(f'{KIND[k].capitalize()} claims:\nshare of explained patients')
    lax = fig.add_subplot(gs[2, j]); lax.axis('off')
    h, l = axes[1, j].get_legend_handles_labels()
    lax.legend(h, l, loc='upper center', bbox_to_anchor=(0.5, 0.93), ncol=1, fontsize=6.5, frameon=False, borderaxespad=0)
axes[1, 1].set_xlabel('Labelled target records used for the certificates', fontsize=8.5)
fig.text(0.5, 0.0, f'Solid: certificates from labelled target records (mean of {DRAWS13} draws; band: middle 95% of draws). '
         'Dashed: source certificates, no target labels.', ha='center', va='bottom', fontsize=6.5, color='0.3')
save(fig, 'fig_shift_recertification', 'Figure 8', '13_shift_recertification.csv, 13_shift_targets.csv, 13_shift.csv')

REC13B, TRN13B = rd('13b_shift_recertification.csv'), rd('13b_shift.csv')
SIZES13B, DRAWS13B = rj('13b_shift.json')['sizes']['recertification_sizes'], rj('13b_shift.json')['sizes']['draws']
STYLE_M = {'BRFSS 2015 test split': ('0.6', 's', 'No shift (BRFSS 2015 test split)'),
           'NHANES, self-reported': ('#0072B2', 'o', 'NHANES, self-reported values'),
           'NHANES, measured BMI and blood pressure': ('#D55E00', 'o', 'NHANES, measured BMI and blood pressure')}
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.3))
for ax, k in zip(axes, (0, 1)):
    for t, (col, mk, lab) in STYLE_M.items():
        s = REC13B[(REC13B.setting == 'measurement') & (REC13B.target == t) & (REC13B.kind == k)].sort_values('m')
        if t != 'BRFSS 2015 test split':
            ax.fill_between(s.m, s.coverage_q025, s.coverage_q975, color=col, alpha=0.13, lw=0)
        ax.plot(s.m, s.coverage, color=col, lw=1.3 if mk == 'o' else 1.6, marker=mk, ms=2.8, label=lab)
        tr = TRN13B[(TRN13B.setting == 'measurement') & (TRN13B.target == t) & (TRN13B.kind == k)].coverage
        if len(tr):
            ax.axhline(float(tr.iloc[0]), color=col, lw=0.9, ls=(0, (4, 2)))
    ax.set_xscale('log')
    ax.xaxis.set_major_locator(FixedLocator(SIZES13B))
    ax.xaxis.set_major_formatter(FixedFormatter([SIZE_LABEL.get(m, str(m)) for m in SIZES13B]))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.grid(color='0.92', lw=0.6)
    ax.set_title(f'{KIND[k].capitalize()} claims', fontsize=8.5)
    ax.set_xlabel('Labelled target records used for the certificates')
axes[0].set_ylabel('Share of explained patients')
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=3, frameon=False, fontsize=7, bbox_to_anchor=(0.5, 1.03))
fig.text(0.5, -0.02, f'Solid: certificates from labelled target records (mean of {DRAWS13B} draws; band: middle 95% of draws). '
         'Dashed: source certificates, no target labels.', ha='center', fontsize=6.5, color='0.3')
fig.tight_layout(rect=(0, 0.02, 1, 0.92))
save(fig, 'fig_shift_measurement', 'Figure A6', '13b_shift_recertification.csv, 13b_shift.csv')

SET_STYLE = {'temporal': ('#0072B2', 'Temporal: later years'), 'geographic': ('#D55E00', 'Census region held out'),
             'state': ('#009E73', 'States with most records')}
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.4))
for i, k in enumerate((0, 1)):
    ax = axes[i]
    d = BOX13[(BOX13.role == 'reference') & (BOX13.box_class == k)].dropna(subset=['domain_auroc'])
    ax.scatter(d.domain_auroc, d.risk_target - d.risk_source, s=5, color='0.65', alpha=0.5, lw=0, label='No shift (source test split)')
    for setting, (col, lab) in SET_STYLE.items():
        d = BOX13[(BOX13.setting == setting) & (BOX13.role == 'target') & (BOX13.box_class == k)].dropna(subset=['domain_auroc'])
        ax.scatter(d.domain_auroc, d.risk_target - d.risk_source, s=5, color=col, alpha=0.5, lw=0, label=lab)
    ax.axhline(0, color='0.35', lw=0.6)
    ax.axvline(0.5, color='0.35', lw=0.6, ls=':')
    ax.set_title(f'{KIND[k].capitalize()} boxes released with source certificates', fontsize=8.5)
    ax.set_xlabel('Within-box domain AUROC (source against target records)')
    ax.yaxis.set_major_locator(MaxNLocator(7))
    ax.grid(color='0.92', lw=0.6)
fig.canvas.draw()
for ax in axes:
    pct_axis(ax, 'y', signed=True)
axes[0].set_ylabel('Label risk in the box: target minus source')
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=4, frameon=False, markerscale=2.5, fontsize=7, bbox_to_anchor=(0.5, 1.03))
fig.tight_layout(rect=(0, 0, 1, 0.93))
save(fig, 'fig_shift_box_diagnostics', 'Figure A7', '13_shift_boxes.csv')

In [ ]:
# Figure 8: subgroups (notebook 14); the policy-target line appears in the legend only when it is drawn
RES14, SIZES14 = rd('14_fairness.csv'), rd('14_group_sizes.csv')
TASKS14 = ['cdc_diabetes', 'brfss2015_heart', 'cardio70k']
plt.rcParams.update({'font.size': 7.5})
STYLE_S = {'SCAX, marginal': ('SCAX, marginal', '#0072B2', 'o', True),
           'SCAX, group-conditional': ('SCAX, group-conditional', '#009E73', 's', True),
           'SCAX, group-conditional, strict': ('SCAX, group-conditional (strict)', '#D55E00', 'D', True),
           'SGR': ('SGR, no explanation', '#555555', 'o', False)}
OFF = dict(zip(STYLE_S, (-0.27, -0.09, 0.09, 0.27)))


def subgroup_figure(kind, fname, paper_figure):
    fig = plt.figure(figsize=(7.4, 6.4))
    gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 0.42, 1], hspace=0.42, wspace=0.1)
    axes = np.array([[fig.add_subplot(gs[i, c]) for c in (0, 1, 3)] for i in (0, 1)])
    policy_line = False
    for j, task in enumerate(TASKS14):
        d = RES14[(RES14.task == task) & (RES14.kind == kind) & (RES14.group != 'All')]
        order = SIZES14[SIZES14.task == task][['attribute', 'group']].values.tolist()
        ypos = {tuple(g): len(order) - 1 - i for i, g in enumerate(order)}
        target, asel = d.claim_target.iloc[0], d.alpha_sel.iloc[0]
        for method, (lab, col, mk, filled) in STYLE_S.items():
            dm = d[d.method == method]
            yy = np.array([ypos[(a, g)] for a, g in zip(dm.attribute, dm.group)]) + OFF[method]
            kw = dict(marker=mk, ms=3.6, ls='none', color=col, mfc=col if filled else 'white', mec=col, mew=0.9)
            axes[0, j].plot(dm.certified_share, yy, **kw)
            ok = dm.accepted > 0
            if kind == 'enrichment':
                v, lo, hi = 1 - dm.error_rate[ok], 1 - dm.error_upper_95[ok], 1 - dm.error_lower_95[ok]
            else:
                v, lo, hi = dm.error_rate[ok], dm.error_lower_95[ok], dm.error_upper_95[ok]
            axes[1, j].errorbar(v, yy[ok.to_numpy()], xerr=[v - lo, hi - v], fmt=mk, ms=3.6, color=col,
                                mfc=col if filled else 'white', mec=col, mew=0.9, elinewidth=0.8, capsize=0)
        axes[1, j].axvline((1 - target) if kind == 'enrichment' else target, color='0.2', ls='--', lw=0.9)
        if not np.isclose(asel, target):
            axes[1, j].axvline((1 - asel) if kind == 'enrichment' else asel, color='0.2', ls=':', lw=0.9)
            policy_line = True
        for i in (0, 1):
            ax = axes[i, j]
            ax.set_yticks(list(ypos.values()))
            same = j > 0 and order == SIZES14[SIZES14.task == TASKS14[j - 1]][['attribute', 'group']].values.tolist()
            ax.set_yticklabels([] if same else [g.replace('$', r'\$') for _, g in ypos.keys()])
            ax.set_ylim(-0.6, len(order) - 0.4)
            ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
            ax.grid(axis='x', color='0.92', lw=0.6); ax.set_axisbelow(True)
            for b_ in range(1, len(order)):
                if order[b_][0] != order[b_ - 1][0]:
                    ax.axhline(len(order) - b_ - 0.5, color='0.8', lw=0.6)
        axes[0, j].set_title(TASK_NAMES[task], fontsize=8.5)
        axes[0, j].set_xlim(left=-0.03 * axes[0, j].get_xlim()[1])
    what = 'negative-outcome' if kind == 'negative outcome' else 'enrichment'
    axes[0, 1].set_xlabel(f'Share of the group given a certified {what} claim')
    axes[1, 1].set_xlabel('Share with the outcome among patients given a claim')
    handles = [Line2D([], [], marker=mk, ls='none', color=col, mfc=col if f_ else 'white', mec=col, ms=4.5, label=lab)
               for lab, col, mk, f_ in STYLE_S.values()]
    handles.append(Line2D([], [], color='0.2', ls='--', lw=0.9, label='Claim target'))
    if policy_line:
        handles.append(Line2D([], [], color='0.2', ls=':', lw=0.9, label='Policy target, where different'))
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.subplots_adjust(left=0.17, right=0.98, top=0.95, bottom=0.14)
    save(fig, fname, paper_figure, '14_fairness.csv, 14_group_sizes.csv')


subgroup_figure('negative outcome', 'fig_subgroups_negative_outcome', 'Figure 9')
subgroup_figure('enrichment', 'fig_subgroups_enrichment', 'Figure A5')

In [ ]:
# Figure 9: runtime (notebook 15)
J15 = rj('15_runtime.json')
DEPTHS15, CELL_CAP = J15['depths'], J15['cell_cap_seconds']
SCALING_TASKS = [t for t in ('brfss2015_heart', 'cardio70k') if t in set(RS[RS.series == 'no regularisation'].task)]
plt.rcParams.update({'font.size': 7.5})
DEPTH_COLOURS = dict(zip([3, 4, 5, 6], ['#9ecae1', '#4292c6', '#08519c', '#08306b']))
fig = plt.figure(figsize=(7.4, 5.4))
gs = fig.add_gridspec(2, max(2, len(SCALING_TASKS)), height_ratios=[1.15, 1], hspace=0.55, wspace=0.28, top=0.84)
secfmt = FuncFormatter(lambda v, _: f'{v:g} s')
for j, t in enumerate(SCALING_TASKS):
    ax = fig.add_subplot(gs[0, j])
    d = RS[RS.task == t]
    for sr, mk, filled in (('frozen settings', 'o', True), ('no regularisation', 's', False)):
        for depth in DEPTHS15[sr]:
            e = d[(d.series == sr) & (d.max_depth == depth)].sort_values('n_trees')
            col = DEPTH_COLOURS[depth]
            ax.plot(e.internal_nodes, e.median_seconds, color=col, lw=1.1, ls='-' if filled else '--', zorder=2)
            ok, lb = e[~e.median_is_lower_bound], e[e.median_is_lower_bound]
            ax.errorbar(ok.internal_nodes, ok.median_seconds, yerr=[ok.median_seconds - ok.q25_seconds, ok.q75_seconds - ok.median_seconds],
                        fmt=mk, ms=3.8, color=col, mfc=col if filled else 'white', mec=col, mew=1.0, elinewidth=0.7, capsize=0, zorder=3)
            ax.plot(lb.internal_nodes, lb.median_seconds, '^', ms=5, color=col, mfc='white', mec=col, mew=1.0, zorder=3)
    f0 = d[d.frozen_model]
    ax.plot(f0.internal_nodes, f0.median_seconds, 'o', ms=10, mfc='none', mec='#D55E00', mew=1.4, zorder=4)
    ax.axhline(5, color='0.3', ls='--', lw=0.9)
    ax.axhline(CELL_CAP, color='0.3', ls=':', lw=0.9)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.yaxis.set_major_formatter(secfmt)
    ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel('Internal nodes in the ensemble (effective size)'); ax.set_title(TASK_NAMES[t], fontsize=8.5)
    ax.grid(color='0.92', lw=0.6); ax.set_axisbelow(True)
    if j == 0:
        ax.set_ylabel('Seconds per distinct cell (median, IQR)')
h = [Line2D([], [], color=DEPTH_COLOURS[k], lw=1.4, label=f'Depth {k}') for k in sorted(DEPTH_COLOURS) if any(k in v for v in DEPTHS15.values())]
h += [Line2D([], [], marker='o', color='0.3', ls='-', ms=4, label='Frozen settings (gamma 2, min. child weight 50)'),
      Line2D([], [], marker='s', color='0.3', mfc='white', ls='--', ms=4, label='No regularisation'),
      Line2D([], [], marker='^', color='0.3', mfc='white', ls='none', ms=5, label='Median is a lower bound (cells stopped)'),
      Line2D([], [], marker='o', color='#D55E00', mfc='none', ls='none', ms=8, label='Frozen model'),
      Line2D([], [], color='0.3', ls='--', lw=0.9, label='RQ6 criterion (5 s)'),
      Line2D([], [], color='0.3', ls=':', lw=0.9, label=f'Cell time cap ({CELL_CAP} s)')]
fig.legend(handles=h, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.995), fontsize=7)
ax = fig.add_subplot(gs[1, :])
labels, segs = [], []
for t in ('cdc_diabetes', 'brfss2015_heart', 'cardio70k'):
    s = RS[(RS.task == t) & RS.frozen_model].iloc[0]
    for stage, name in (('deletion', 'deletion (AXps)'), ('inflation', 'inflation')):
        pre, sea, tot = s[f'mean_{stage}_precheck_seconds'], s[f'mean_{stage}_search_seconds'], s[f'mean_{stage}_seconds']
        labels.append(f'{TASK_NAMES[t]}, {name}'); segs.append((pre, sea, max(0.0, tot - pre - sea)))
segs = np.array(segs); yy = np.arange(len(labels))[::-1]
left = np.zeros(len(labels))
for k, (name, col) in enumerate([('Interval pre-check', '#56B4E9'), ('Exact search', '#0072B2'), ('Other (memo, bookkeeping)', '#BBBBBB')]):
    ax.barh(yy, segs[:, k], left=left, color=col, height=0.62, label=name, edgecolor='white', linewidth=0.6)
    left += segs[:, k]
ax.set_yticks(yy); ax.set_yticklabels(labels)
ax.set_xlabel('Mean seconds per distinct cell, frozen models, by step')
ax.xaxis.set_major_formatter(secfmt)
ax.legend(frameon=False, loc='lower right', fontsize=7)
ax.grid(axis='x', color='0.92', lw=0.6); ax.set_axisbelow(True)
save(fig, 'fig_runtime', 'Figure 10', '15_runtime.csv, 15_runtime.json')

In [ ]:
# Figures S9 and S10: ablations (notebooks 12 and 12b)
plt.rcParams.update({'font.size': 8})
NAMES2 = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease'}
COL2 = {'cdc_diabetes': ('#0072B2', 'o'), 'brfss2015_heart': ('#D55E00', 's')}
FACTORS = ['prior', 'explanation construction', 'candidates M', 'calibration size']
TITLE = {'prior': 'Prior', 'explanation construction': 'Explanation construction', 'candidates M': 'Number of candidate orders',
         'calibration size': 'Size of calibration split 1'}
heights = [AB[AB.factor == f_].level.nunique() + 1.2 for f_ in FACTORS]
fig, axes = plt.subplots(len(FACTORS), 2, figsize=(7.4, 8.2), sharex='col', gridspec_kw={'height_ratios': heights})
for i, fac in enumerate(FACTORS):
    levels = list(dict.fromkeys(AB[AB.factor == fac].level))
    for j, kind in enumerate(KIND.values()):
        ax = axes[i, j]
        for k, task in enumerate(('cdc_diabetes', 'brfss2015_heart')):
            g = AB[(AB.factor == fac) & (AB.kind == kind) & (AB.task == task)].set_index('level').reindex(levels)
            yv = np.arange(len(levels))[::-1] + (0.15 if k == 0 else -0.15)
            col, mk = COL2[task]
            ax.errorbar(g.explained_share, yv, xerr=[g.explained_share - g.explained_lo, g.explained_hi - g.explained_share],
                        fmt=mk, color=col, ms=4.5, lw=1, capsize=0, label=NAMES2[task])
        frozen = set(AB[(AB.factor == fac) & AB.frozen].level)
        ax.set_yticks(np.arange(len(levels))[::-1])
        ax.set_yticklabels([l_.replace('lambda', 'λ') + (' (frozen)' if l_ in frozen else '') for l_ in levels] if j == 0 else [])
        ax.set_ylim(-0.7, len(levels) - 0.3)
        ax.grid(axis='x', color='0.9', lw=0.6)
        if i == 0:
            ax.set_title(f'{kind.capitalize()} claims', fontsize=9)
        if j == 0:
            ax.text(-0.02, 1.02, TITLE[fac], transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, fontweight='bold')
for ax in axes[-1]:
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
axes[-1, 0].set_xlim(0, 0.75); axes[-1, 1].set_xlim(0, 0.45)
fig.supxlabel('Share of explained test patients given an explanation', x=0.62, fontsize=8.5)
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.6, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.97))
save(fig, 'fig_ablations_frozen_model', 'Figure A3', '12_ablations_summary.csv')

N_TEST = rj('12b_model_ablations.json')['test_patients']
levels = list(dict.fromkeys(AB2.level))


def display(l_):
    if l_.startswith('frozen'):
        return 'Frozen model (XGBoost, depth 4)'
    if l_.startswith('max_bin'):
        return f"{l_.split()[1]} threshold bins"
    return {'clinically signed': 'Monotone constraints', 'random forest': 'Random forest'}.get(l_, l_[0].upper() + l_[1:])


fig, axes = plt.subplots(1, 2, figsize=(7.4, 4.6), sharey=True)
for j, kind in enumerate(KIND.values()):
    ax = axes[j]
    for k, task in enumerate(('cdc_diabetes', 'brfss2015_heart')):
        g = AB2[(AB2.task == task) & (AB2.kind == kind)].set_index('level').reindex(levels)
        yv = np.arange(len(levels))[::-1] + (0.15 if k == 0 else -0.15)
        col, mk = COL2[task]
        ax.errorbar(g.coverage, yv, xerr=[g.coverage - g.explained_lo, g.explained_hi - g.coverage], fmt=mk, color=col,
                    ms=4.5, lw=1, capsize=0, label=NAMES2[task])
        ax.axvline(g.loc[levels[0], 'coverage'], color=col, lw=0.6, ls=':')
    ax.set_yticks(np.arange(len(levels))[::-1]); ax.set_yticklabels([display(l_) for l_ in levels])
    ax.set_title(f'{kind.capitalize()} claims', fontsize=9)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.grid(axis='x', color='0.9', lw=0.6)
fig.supxlabel(f'Share of {N_TEST} explained test patients given an explanation (dotted: frozen model)', fontsize=8.5)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.6, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, 'fig_ablations_model_factors', 'Figure A4', '12b_model_ablations.csv')

In [ ]:
# Figure S11: self-reported against measured values in NHANES (notebook 03), drawn from a summary table
NH_FILE = paths.DERIVED / 'nhanes' / 'nhanes_2015_2018.parquet'
if NH_FILE.exists():
    check_data(NH_FILE, rj('03_nhanes_uci.json')['nhanes']['parquet_sha256'], 'NHANES table')
    NH = pd.read_parquet(NH_FILE)
    diff = NH.BMI_self_exact - NH.BMI_measured_exact
    edges = np.arange(-10, 10.5, 0.5)
    counts, _ = np.histogram(diff.clip(-10, 10), bins=edges)
    rows = [{'table': 'bmi_difference_histogram', 'left': float(a), 'right': float(b), 'adults': int(c)} for a, b, c in zip(edges[:-1], edges[1:], counts)]
    rows.append({'table': 'bmi_difference_median', 'value': float(diff.median()), 'adults': int(diff.notna().sum())})
    bands = pd.cut(NH.age_years, [19, 39, 59, 200], labels=['20-39', '40-59', '60+'])
    for s_, m_ in (('HighBP', 'HighBP_measured'), ('HighChol', 'HighChol_measured')):
        g = NH.assign(band=bands).dropna(subset=[m_]).groupby('band', observed=True)[[s_, m_]].agg(['mean', 'size'])
        for band in g.index:
            rows.append({'table': f'{s_}_by_age', 'band': band, 'self_reported': float(g.loc[band, (s_, 'mean')]),
                         'measured': float(g.loc[band, (m_, 'mean')]), 'adults': int(g.loc[band, (s_, 'size')])})
    pd.DataFrame(rows).to_csv(RES / '16_nhanes_measurement_summary.csv', index=False)
NS = rd('16_nhanes_measurement_summary.csv')
SELF, MEAS = '#0072B2', '#D55E00'
plt.rcParams.update({'font.size': 8})
fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.7))
ax = axes[0]
hst = NS[NS.table == 'bmi_difference_histogram']
ax.bar(hst.left, hst.adults, width=hst.right - hst.left, align='edge', color='0.55', edgecolor='white', linewidth=0.3)
medv = float(NS[NS.table == 'bmi_difference_median'].value.iloc[0])
ax.axvline(0, color='0.2', lw=0.8, ymax=0.8)
ax.axvline(medv, color=MEAS, lw=1.2, ls='--', ymax=0.8, label=f'median {medv:+.1f}'.replace('-', '−'))
ax.set_xlabel('BMI: self-reported minus measured (kg/m²)'); ax.set_ylabel('Adults')
ax.set_ylim(top=ax.get_ylim()[1] * 1.2)
ax.legend(frameon=False, fontsize=7, loc='upper left'); ax.set_title('BMI', fontsize=8.5)
for ax, (s_, title) in zip(axes[1:], [('HighBP', 'High blood pressure'), ('HighChol', 'High cholesterol')]):
    g = NS[NS.table == f'{s_}_by_age']
    x = np.arange(len(g))
    ax.bar(x - 0.2, g.self_reported, 0.38, color=SELF, label='Ever told (self-reported)')
    ax.bar(x + 0.2, g.measured, 0.38, color=MEAS, label='Measured at examination')
    ax.set_xticks(x, list(g.band)); ax.set_xlabel('Age (years)'); ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.set_title(title, fontsize=8.5)
    ax.grid(axis='y', color='0.92', lw=0.6)
axes[1].set_ylabel('Share of adults')
h, l = axes[1].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=2, frameon=False, fontsize=7, bbox_to_anchor=(0.62, 1.04))
fig.tight_layout(rect=(0, 0, 1, 0.92))
save(fig, 'fig_nhanes_measurement', 'Figure A8', '16_nhanes_measurement_summary.csv (from the NHANES table of notebook 03)')
FIGS = pd.DataFrame(FIGS)
missing = sorted({p.stem for p in FIG.glob('fig_*.png')} - {f_.split('/')[-1][:-4] for f_ in FIGS.file})
print(FIGS.to_string(index=False)); print('figures in the folder not regenerated:', missing or 'none')
assert not missing, 'a figure in results/figures was not regenerated from CSV'

## Claims register

In [ ]:
CL = paper.Claims()
mt = MAIN.copy()


def row(task, fmy, kind):
    return mt[(mt.task == task) & (mt.family == fmy) & (mt.kind == kind)].iloc[0]


# Soundness
CL.add('S1', 'Soundness (Theorem 1 in practice, gate G2)', 'No disagreement was recorded between the encodings, brute force, the two exact '
       'checks, or exported and native predictions, in any check recorded in the cited files.', GATES[GATES.gate == 'G2 soundness'].sources.iloc[0],
       sum(DIS.values()) == 0)
tot_el, tot_sd = sum(v['eligible_boxes'] for v in VER.values()), sum(v['sound'] for v in VER.values())
CL.add('S2', 'Soundness (Theorem 1 in practice, gate G2)', f'Every box that could be released at the loosest grid target for the frozen '
       f'XGBoost models ({tot_el:,} boxes over three tasks) was confirmed sound by the second exact implementation; none was undecided.',
       '08_certificates.json, 09d_results.json', tot_el == tot_sd)
CL.add('S3', 'Soundness (Theorem 1 in practice, gate G2)', f"Brute-force enumeration on small ensembles: {g2['brute_force_errors']} errors in "
       f"{g2['brute_force_checks']:,} checks.", '06_encoding_tests.json', g2['brute_force_errors'] == 0)

# Validity
vs_mix, vs_occ, vs_nai = V[V.prior == MIX], V[V.prior == 'occam'], V[V.prior == 'naive']
CL.add('V1', 'Validity (E2, RQ1, gate G4)', f'No SCAX configuration showed excess box-certificate violations ({len(VS)} configurations, Occam '
       f'and mixture priors); the largest violation frequency was {VS.V1_rate.max():.3f}.', '10_validity.csv' + (', 10b_validity_more_repetitions.csv' if MORE.exists() else ''), (VS.V1_verdict != 'excess').all())
CL.add('V2', 'Validity (E2, RQ1, gate G4)', f'No configuration showed excess policy violations; the largest frequency was {VS.V2_rate.max():.3f} '
       f'against delta_2 = 0.025.', '10_validity.csv' + (', 10b_validity_more_repetitions.csv' if MORE.exists() else ''), (VS.V2_verdict != 'excess').all())
CL.add('V3', 'Validity (E2, RQ1, gate G4)', f"Per-box bounds without the prior showed excess violations in {int((vs_nai.V1_verdict == 'excess').sum())} "
       f"of {len(vs_nai)} configurations (Proposition 3).", '10_validity.csv', (vs_nai.V1_verdict == 'excess').sum() > 0)
inc_mix = vs_mix[vs_mix.V1_verdict == 'inconclusive']
more_reps = vs_mix[vs_mix.V1_reps > vs_mix.reps]
if len(inc_mix):
    v4 = (f"{len(inc_mix)} of {len(vs_mix)} were inconclusive (at "
          + ', '.join(f'{int(r):,}' for r in sorted(inc_mix.V1_reps.unique())) + ' repetitions), each with a point estimate at or below delta_1')
else:
    v4 = f"every configuration was 'low enough' ({int((vs_mix.V1_verdict == 'low enough').sum())} of {len(vs_mix)})"
    if len(more_reps):
        v4 += (f", {len(more_reps)} of them after {int(more_reps.V1_reps.max()):,} repetitions because {int(more_reps.reps.max())} "
               'had left them inconclusive')
CL.add('V4', 'Validity (E2, RQ1, gate G4)', f"With the Occam prior every configuration was 'low enough' "
       f"({int((vs_occ.V1_verdict == 'low enough').sum())} of {len(vs_occ)}); with the mixture prior {v4}.",
       '10_validity.csv' + (', 10b_validity_more_repetitions.csv' if MORE.exists() else ''),
       (vs_occ.V1_verdict == 'low enough').all() and bool((vs_mix[vs_mix.V1_verdict == 'inconclusive'].V1_rate <= vs_mix[vs_mix.V1_verdict == 'inconclusive'].delta1 + 1e-12).all()))
ov = p4[p4.prior == MIX].sort_values('M')
CL.add('V5', 'Validity (E2, RQ1, gate G4)', f"Overlap problem on P4 ({max(N1_LIST):,} calibration records, delta_1 = 0.025): every released box "
       f"was within target while the released patients were above alpha_sel in up to {(ov.overlap / ov.reps).max():.0%} of repetitions "
       f"(mixture prior), and the policy test certified such a policy in at most {(ov.V2 / ov.reps).max():.1%}.", '10_validity.csv',
       (ov.overlap / ov.reps).max() > 0 and (ov.V2 / ov.reps).max() <= 0.025)

# Main results
pol = mt[mt.task != 'uci_heart']
cert = pol[pol.policy_certified == True]
CL.add('M1', 'Main results (E1, RQ2, RQ3)', f"The release policy was certified in {len(cert)} of {len(pol)} task, model and claim units with a policy "
       f"test: " + '; '.join(f"{TASK_NAMES[t]} {int((g.policy_certified == True).sum())} of {len(g)}" for t, g in pol.groupby('task', sort=False)) + '.',
       '09b_main.csv, 09e_cardio_main.csv', bool(((pol.U_sel <= pol.alpha_sel) == pol.policy_certified.astype(bool)).all()))
CL.add('M2', 'Main results (E1, RQ2, RQ3)', 'In every unit whose policy was certified, the share with the outcome among released test patients '
       'met the policy target (at most alpha_sel for negative-outcome claims, at least 1 - alpha_sel for enrichment claims).', '09b_main.csv, 09e_cardio_main.csv', bool((cert.error_rate <= cert.alpha_sel + 1e-12).all()))
for task in ('brfss2015_heart', 'cdc_diabetes', 'cardio70k'):
    for kind in KIND.values():
        r = row(task, 'xgboost', kind)
        word = 'negative-outcome' if kind == 'negative outcome' else 'enrichment'
        CL.add(f"M-{task}-{word}", 'Main results (E1, RQ2, RQ3)',
               f"{TASK_NAMES[task]}, frozen XGBoost, {claim_text(kind, r.claim_target)}: policy {policy_text(kind, r.policy_certified, r.U_sel, r.alpha_sel)}; "
               f"{pct_ci(r.certified_share, r.certified_share_lo, r.certified_share_hi)} of explained test patients received a certified claim, "
               f"and {pct_ci(r.outcome_share, r.outcome_share_lo, r.outcome_share_hi)} of them had the outcome.", r.source,
               (r.U_sel <= r.alpha_sel) == bool(r.policy_certified))
flags = sum(int(rd(f_).estimate_above_target.sum()) for f_ in ('09_box_consistency.csv', '09d_box_consistency.csv', '09b_box_consistency.csv',
                                                                '09e_cardio_box_consistency.csv'))
CL.add('M3', 'Main results (E1, RQ2, RQ3)', f'No released box had a test error significantly above its target on the full test split '
       f'(Bonferroni-corrected lower limits; {flags} flags over all models).',
       '09_box_consistency.csv, 09d_box_consistency.csv, 09b_box_consistency.csv, 09e_cardio_box_consistency.csv', flags == 0)
ut = UT
neg_uci, enr_uci = ut[ut.kind == 0], ut[ut.kind == 1]
prec = 1 - enr_uci.error_rate
CL.add('M4', 'Main results (E1, RQ2, RQ3)', f"UCI Heart (each site held out, three models): no negative-outcome claim was certified at alpha_0 = "
       f"{pct(ALPHA_H[0])}; enrichment claims were certified at every held-out site for every model ({pct(enr_uci.coverage.min())} to "
       f"{pct(enr_uci.coverage.max())} of patients), with {pct(prec.min(), 0)} to {pct(prec.max(), 0)} of them having angiographic disease "
       f"and no claim refuted.", '09e_uci_main.csv',
       (neg_uci.accepted == 0).all() and (enr_uci.accepted > 0).all() and (enr_uci.refuted_share == 0).all())
mx = mt[(mt.kind == 'negative outcome') & mt.auroc.notna()]
CL.add('M5', 'Main results (E1, RQ2, RQ3)', 'Test AUROC of the underlying models: ' + '; '.join(
    f"{r.task_name} {r.family_name} {r.auroc:.3f}" for r in mx.itertuples()) + '.', '09b_model_metrics.csv, 09e_cardio_model_metrics.csv', True)
CL.add('M6', 'Main results (E1, RQ2, RQ3)', f"Bootstrap intervals from the saved releases and from the saved counts agree for the {len(CHECK)} "
       'XGBoost models, and the saved releases reproduce the saved counts.', '09_test_releases.csv, 09d_test_releases.csv, 16_bootstrap_check.csv',
       bool(CHECK.same_distribution.all()))

# Baselines
sc = BS[BS.method == 'SCAX']
ax_m = BS[BS.method == 'Uncertified AXp, matched coverage']
CL.add('B1', 'Baselines (E1, E3)', 'SCAX released no box whose test error was above its target in either task or claim kind (mean over '
       '20 calibration re-draws).', '11_baselines_summary.csv', (sc.above_target_share == 0).all())
CL.add('B2', 'Baselines (E1, E3)', f"Uncertified AXps at SCAX's coverage released boxes above their target for {pct(ax_m[ax_m.kind == 'negative outcome'].above_target_share.min())} "
       f"to {pct(ax_m[ax_m.kind == 'negative outcome'].above_target_share.max())} of negative-outcome and {pct(ax_m[ax_m.kind == 'enrichment'].above_target_share.min())} "
       f"to {pct(ax_m[ax_m.kind == 'enrichment'].above_target_share.max())} of enrichment patients.", '11_baselines_summary.csv',
       (ax_m.above_target_share > 0).all())
rdict = BS[BS.method == 'Rule dictionary']
CL.add('B3', 'Baselines (E1, E3)', f"None of the rules of the training rule dictionary used for releases is sound for the deployed model "
       f"(sound share {pct(rdict.sound_share.max())}).", '11_baselines_summary.csv', (rdict.sound_share.fillna(0) == 0).all())
sy_s, sy_n, sy_e = SY[SY.method == 'SCAX'], SY[SY.method == 'Per-box bound without prior'], SY[SY.method == 'Empirical error rule']
CL.add('B4', 'Baselines (E1, E3)', f"On the planted rule P4 SCAX made no false claim in any repetition at either target pair; per-box bounds "
       f"without the prior made at least one false claim in {pct(sy_n.any_false_claim.max())} and {pct(sy_n.any_false_claim.min())} of repetitions, "
       f"the empirical rule in {pct(sy_e.any_false_claim.min())}.", '11_synthetic_summary.csv', (sy_s.any_false_claim == 0).all())

# Ablations
def share(task, factor, level, kind):
    return float(AB[(AB.task == task) & (AB.factor == factor) & (AB.level == level) & (AB.kind == kind)].explained_share.iloc[0])


fz = 'every sound box on the inflation path'
CL.add('A1', 'Ablations (E3)', 'Keeping every sound box on the inflation path, rather than the AXps only, raised the certified share from ' + '; '.join(
    f"{pct(share(t, 'explanation construction', 'AXps only', k))} to {pct(share(t, 'explanation construction', fz, k))} ({NAMES2[t]}, {k})"
    for t in ('cdc_diabetes', 'brfss2015_heart') for k in KIND.values()) + '.', '12_ablations_summary.csv',
    all(share(t, 'explanation construction', fz, k) > share(t, 'explanation construction', 'AXps only', k) for t in NAMES2 for k in KIND.values()))
CL.add('A2', 'Ablations (E3)', 'The three selection rules gave the same certified share for both tasks and kinds (they change which box a '
       'patient receives, not whether one is certified).', '12_ablations_summary.csv',
       AB[AB.factor == 'selection rule'].groupby(['task', 'kind']).explained_share.nunique().eq(1).all())
CL.add('A3', 'Ablations (E3)', 'The certified share grew with the number of candidate orders M (1 to 10) for both tasks and kinds.',
       '12_ablations_summary.csv', all(AB[(AB.factor == 'candidates M') & (AB.task == t) & (AB.kind == k)].explained_share.is_monotonic_increasing
                                       for t in NAMES2 for k in KIND.values()))
CL.add('A4', 'Ablations (E3)', f'No model variant (family, depth, trees, bins, monotone constraints) released a box refuted on the test split.',
       '12b_model_ablations.csv', (AB2.refuted_share == 0).all())

# Shift
SHT = SH[SH.role == 'target']
CL.add('H1', 'Shift (E4, RQ4)', f"Source certificates transported to {SHT.target.nunique()} target domains (later years, held-out regions, states, "
       f"NHANES with self-reported and measured values, UCI sites) had no refuted claim and no refuted certificate.",
       '13_shift.csv, 13b_shift.csv', (SHT.refuted_share == 0).all() and (SHT.certificate_refuted_share == 0).all())
CL.add('H2', 'Shift (E4, RQ4)', f"The share of released patients whose box had a target error estimate above its target was at most "
       f"{pct(SHT.above_target_share.max())} ({SHT.loc[SHT.above_target_share.idxmax(), 'target']}, {SHT.loc[SHT.above_target_share.idxmax(), 'kind_name']}).",
       '13_shift.csv, 13b_shift.csv', True)

# Subgroups
mg = FS[FS.method == 'SCAX, marginal']; gc = FS[FS.method == 'SCAX, group-conditional']; st = FS[FS.method == 'SCAX, group-conditional, strict']
miss = mg[mg.groups_above_claim_target > 0]
CL.add('SG1', 'Subgroups (E5, RQ5)', f"Marginal SCAX left at least one group whose observed share with the outcome missed the claim target in "
       f"{len(miss)} of {len(mg)} task, attribute and kind combinations.", '14_fairness_summary.csv', True)
CL.add('SG2', 'Subgroups (E5, RQ5)', 'Group-conditional SCAX (standard) left no group missing the negative-outcome claim target.',
       '14_fairness_summary.csv', (gc[gc.kind == 'negative outcome'].groups_above_claim_target == 0).all())
CL.add('SG3', 'Subgroups (E5, RQ5)', 'The strict version left no group missing the claim target, for either kind.', '14_fairness_summary.csv',
       (st.groups_above_claim_target == 0).all())
pp = lambda v: f'{100 * v:.1f}'
CL.add('SG4', 'Subgroups (E5, RQ5)', f"Cost in certified share against marginal SCAX: standard {pp(gc.coverage_cost_vs_marginal.min())} to "
       f"{pp(gc.coverage_cost_vs_marginal.max())} percentage points, strict {pp(st.coverage_cost_vs_marginal.min())} to "
       f"{pp(st.coverage_cost_vs_marginal.max())} percentage points of explained test patients.", '14_fairness_summary.csv', True)

# Runtime
CL.add('R1', 'Runtime (E6, RQ6, gate G5)', f"Production runtime of the main results: median {RP.median_seconds.min():.2f} to {RP.median_seconds.max():.2f} "
       f"seconds per distinct cell, 95th percentile at most {RP.p95_seconds.max():.2f} s, maximum {RP.max_seconds.max():.1f} s, no search stopped "
       f"by the node budget ({len(RP)} models).", '15_runtime_production.csv', (RP.search_unknown == 0).all() and RP.median_seconds.max() <= 30)
rq = J15['RQ6']
CL.add('R2', 'Runtime (E6, RQ6, gate G5)', f"With the frozen settings every model up to 100 trees and depth 6 had a median of at most "
       f"{rq['frozen settings']['largest_median_seconds']:.2f} s (RQ6 criterion 5 s); without regularisation "
       f"{len(rq['no regularisation']['models_above_5_seconds'])} of {int(((RS.series == 'no regularisation') & (RS.n_trees <= 100)).sum())} models up to 100 trees were above 5 s.",
       '15_runtime.json, 15_runtime.csv', rq['frozen settings']['met'])
fr = RS[RS.frozen_model]
CL.add('R3', 'Runtime (E6, RQ6, gate G5)', 'The frozen 100-round models have ' + '; '.join(
    f"{TASK_NAMES[r.task]} {int(r.trees_with_a_split)} trees with a split ({int(r.internal_nodes)} internal nodes)" for r in fr.itertuples()) + '.',
    '15_runtime.csv', True)

# Survey-weighted sensitivity and explanation cards
if len(WEIGHTED):
    for r in WEIGHTED.itertuples():
        word = 'negative-outcome' if r.kind == 'negative outcome' else 'enrichment'
        CL.add(f"W-{r.task}-{word}", 'Survey-weighted sensitivity (plan Section 6.2)',
               f"{r.task_name}, {word} claims, frozen XGBoost: with the BRFSS survey weights, {pct_ci(r.certified_share_weighted, r.certified_share_weighted_lo, r.certified_share_weighted_hi)} "
               f"of explained test patients received a certified claim (unweighted {pct(r.certified_share)}), and {pct_ci(r.outcome_share_weighted, r.outcome_share_weighted_lo, r.outcome_share_weighted_hi)} "
               f"of them had the outcome (unweighted {pct(r.outcome_share)}).", '16_weighted_sensitivity.csv, 02_brfss_rebuild.json', r.patients == 5000,
               note='descriptive; certificates are unweighted')
for r in CARDS.itertuples():
    word = 'negative-outcome' if r.kind == 'negative outcome' else 'enrichment'
    ok = r.bound <= r.claim_target
    CL.add(f"C-{r.task}-{word}", 'Explanation cards (introduction)',
           f"{r.task_name}, {word} claim given to {r.explained_test_patients_given_it} explained test patients: '{r.explanation}'; "
           f"{r.calibration_records:,} calibration records, {r.claim}.", '16_example_cards.csv, ' + ('09d_test_releases.csv' if r.task == 'cardio70k' else '09_test_releases.csv'), ok)

# Gates
for r in GATES.itertuples():
    CL.add(f"Gate-{r.gate.split()[0]}", 'Gates', f'{r.gate}: {r.status}.', r.sources, not r.status.startswith('failed'))
CLAIMS = CL.frame()
print(CLAIMS[['id', 'holds']].T.to_string(header=False))
print(f"{int(CLAIMS.holds.sum())} of {len(CLAIMS)} checks hold")

## Write the outputs

In [ ]:
MAIN.to_csv(RES / '16_table_main.csv', index=False)
UCI_SITES.to_csv(RES / '16_table_uci_sites.csv', index=False)
TV.to_csv(RES / '16_table_validity.csv', index=False)
GATES.to_csv(RES / '16_gates.csv', index=False)
CHECK.to_csv(RES / '16_bootstrap_check.csv', index=False)
CAND.to_csv(RES / '16_candidate_stats.csv', index=False)
WEIGHTED.to_csv(RES / '16_weighted_sensitivity.csv', index=False)
CARDS.to_csv(RES / '16_example_cards.csv', index=False)
FIGS.to_csv(RES / '16_figures.csv', index=False)
MANIFEST.to_csv(RES / '16_manifest.csv', index=False)
VERSIONS.to_csv(RES / '16_versions.csv', index=False)
(RES / '16_requirements_lock.txt').write_text(LOCK_TEXT)
md = ['# Paper tables', '', 'Written by notebook 16 from the committed results files; every table names its source.', '']
md += [f'## Gates G0 to G5', '', md_table(GATES[['gate', 'status', 'evidence']]), '']
for title, src, body in SECTIONS:
    md += [f'## {title}', '', f'Source: {src}', '', body, '']
md += ['## Figures', '', md_table(FIGS), '']
(RES / '16_tables.md').write_text('\n'.join(md))
(RES / '16_claims_check.md').write_text(CL.markdown(
    'Claims check', 'Every numerical claim planned for the paper, written by notebook 16 from the results file it cites, with a check '
    'computed from that file. A claim whose check does not hold is not used in the paper as stated.'))
write_json(RES / '16_paper_results.json', {
    'seed': SEED, 'bootstrap_resamples': B, 'strict_data_hashes': STRICT, 'data_hashes_match': DATA_OK,
    'gates': GATES.to_dict('records'), 'claims_hold': int(CLAIMS.holds.sum()), 'claims': len(CLAIMS),
    'claims_not_holding': CLAIMS[~CLAIMS.holds].id.tolist(), 'figures': len(FIGS), 'results_files_read': len(MANIFEST),
    'main_table': MAIN.to_dict('records')})
print('written: results/16_table_main.csv, 16_table_uci_sites.csv, 16_table_validity.csv, 16_gates.csv, 16_bootstrap_check.csv, '
      '16_candidate_stats.csv, 16_weighted_sensitivity.csv, 16_example_cards.csv, 16_figures.csv, 16_manifest.csv, 16_versions.csv, 16_requirements_lock.txt, 16_tables.md, '
      '16_claims_check.md, 16_nhanes_measurement_summary.csv, 16_paper_results.json; every figure regenerated')
print('gates:', dict(zip(GATES.gate, GATES.status)))
print('claims not holding:', CLAIMS[~CLAIMS.holds].id.tolist() or 'none')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "16 (version 2): survey-weighted sensitivity analysis (plan 6.2), explanation cards, manuscript Figures 1 and 2 and figure numbering, claim V4 wording; tables, figures and claims check regenerated"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)